# ⛰️ 04 · Calculus and optimization: how machines learn
### Derivatives, gradients, gradient descent, convexity, the matrix calculus of least squares, Lagrange multipliers and coordinate descent

**BiologyToML** · Level 1 · 02-620 core · Part A — Foundations: from biology to data · Module 04 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/04_calculus_and_optimization.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.optimize`), pandas, Matplotlib |
| **Prerequisites** | Modules 01–03 (NumPy; probability and the Gaussian; vectors, matrices, $X^\top X$, eigenvectors). No calculus course needed. |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | The calculus and optimisation prerequisite of 02-620 (CMU 02-680) and the optimisation the lectures use: setting the derivative of a log-likelihood to zero (maximum likelihood); the conditional log-likelihood of logistic regression, its gradient, concavity and gradient ascent with a step size ("Mount Fuji instead of the Himalayas"); the matrix-calculus derivation of the normal equations and of ridge regression (regression); Lagrange multipliers, the primal–dual toy problem and the dual of the support vector machine with its support vectors (SVM); PCA as a constrained optimisation (dimensionality reduction); pathwise coordinate descent for the lasso |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED** by code. The prostate-cancer table (gradient descent and coordinate descent on least
> squares) comes from the toolkit's `simulate_prostate_clinical`, shaped like Stamey *et al.* (1989); the leukaemia
> expression data (a logistic log-likelihood surface for one gene; a two-gene covariance for the PCA constraint problem)
> from `simulate_leukemia_expression`, shaped like Golub *et al.* (1999) — the values are not the studies' own. The
> bacterial growth curve, the replicate measurements, the genotypes of the underflow demo, the two-population mixture
> and the tumour toy of the SVM section are simulated as well and labelled as such.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** a derivative as a slope, **apply** the power, exponential, logarithm, product and chain rules, and **check** every derivative numerically with finite differences — including $\sigma'(z) = \sigma(z)(1-\sigma(z))$ for the logistic function.
2. **Derive** by setting a derivative to zero that the mean minimises squared error and the median minimises absolute error, and **interpret** the difference for replicate measurements with an outlier.
3. **Explain** why machine learning maximises *log*-likelihoods, using a numerical underflow experiment.
4. **Compute** partial derivatives and gradients, **read** contour plots, and **implement** gradient descent/ascent with a stopping rule; **diagnose** step sizes that are too large or too small — the training algorithm of logistic regression.
5. **Distinguish** convex from non-convex objectives ("Mount Fuji instead of the Himalayas") and **use** random restarts against local optima.
6. **Derive** the gradient of $(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w)$ with two vector-calculus rules, and **verify** the normal equations (and their ridge version) on data.
7. **Solve** constrained problems with Lagrange multipliers: the primal/dual toy problem, the dual of the support vector machine with its support vectors, and the PCA problem $\max \mathbf u^\top S\mathbf u$ s.t. $\mathbf u^\top\mathbf u = 1$.
8. **Implement** coordinate descent and **connect** it to the lasso (Module 07) and Gibbs sampling (Module 21).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, simulate the data | 5 min |
| 1 · Derivatives as slopes | growth rate of a culture, the rules, finite differences, $\sigma' = \sigma(1-\sigma)$ | 20 min |
| 2 · Minimising by setting the derivative to zero | mean vs median for replicate wells with an outlier | 15 min |
| 3 · Logs turn products into sums | underflow of a 1,000-genotype likelihood; `logsumexp` | 10 min |
| 4 · Gradients and gradient descent | 1-D step sizes, contour plots, paths on the prostate loss surface | 25 min |
| 5 · Convex vs non-convex | chords, Hessians, a logistic "Mount Fuji", a mixture "Himalaya", restarts | 15 min |
| 6 · Vector calculus for regression | $\partial(\mathbf a^\top\mathbf w)/\partial\mathbf w$, $\partial(\mathbf w^\top A\mathbf w)/\partial\mathbf w$, the normal equations step by step; the ridge (MAP) version | 20 min |
| 7 · Constrained optimisation | Lagrange multipliers: PCA as an eigenproblem; the primal/dual toy; the SVM dual and its support vectors | 30 min |
| 8 · Coordinate descent | one weight at a time; staircase paths | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

What does it mean for a machine to **learn**? In every 02-620 lecture the answer has the same shape: write down a model
with unknown numbers (parameters) and choose the parameters that make the model agree best with the data. "Agree best"
is measured by a score — a likelihood to maximise or an error to minimise — and learning is a search for the top of
a hill or the bottom of a valley in parameter space.

Biologists do this every day without calling it machine learning: fitting a growth rate to optical-density readings,
a dissociation constant to a binding curve, an EC50 to a dose–response experiment, an effect size to a GWAS SNP. The
same machinery trains logistic regression on leukaemia expression profiles (Module 09), finds the maximum-margin
boundary of a support vector machine (Module 11), the principal axes of human genetic variation (Module 14) and the
weights of a neural network (Module 22).

Picture a hiker in thick fog who wants to reach the lowest point of a valley. She cannot see the landscape, but she can
feel the **slope** under her feet and step downhill. Calculus is the mathematics of that slope; optimisation is the art
of stepping well. This notebook teaches both, from the definition of a derivative to the Lagrange multipliers of
support vector machines and PCA.

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: every table here is generated by code.
> The prostate-cancer table comes from the toolkit's `simulate_prostate_clinical`, which imitates the study of
> Stamey *et al.* (1989) that 02-620 uses for regression (97 men, 8 clinical measurements, log PSA as outcome); the
> leukaemia expression matrix comes from `simulate_leukemia_expression`, which imitates Golub *et al.* (1999). Because
> we know the parameters the simulators used, we can check whether an optimiser lands near the truth. None of the
> numbers below are the papers' numbers.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They generate the SIMULATED prostate-cancer table (shaped like Stamey *et al.* 1989) and leukaemia expression data (shaped like Golub *et al.* 1999), compute the logistic function without overflow, compute log-sums of exponentials stably, and z-score columns. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def simulate_prostate_clinical(seed: int = 1989) -> pd.DataFrame:
    """Synthetic clinical data shaped like the prostate-cancer study of Stamey et al. (1989).

    97 men about to undergo radical prostatectomy; 8 clinical measurements — log cancer volume
    (``lcavol``), log prostate weight (``lweight``), ``age``, log benign prostatic hyperplasia
    amount (``lbph``), seminal-vesicle invasion (``svi``, 0/1), log capsular penetration
    (``lcp``), Gleason score (``gleason``) and % Gleason scores 4 or 5 (``pgg45``) — and the
    outcome ``lpsa`` = log(prostate-specific antigen).  Marginals and correlations mimic the
    original data (``lbph`` and ``lcp`` have a floor at log 0.25 = −1.386); ``lpsa`` is generated
    from the standardised predictors with the least-squares coefficients reported for the
    original study by Hastie, Tibshirani & Friedman (2009, Table 3.2) plus noise (sd 0.71).
    The boolean ``train`` marks a 67/30 training/test split.
    """
    rng = np.random.default_rng(seed)
    n = 97
    names = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
    R = np.array([[1.00, 0.29, 0.19, 0.01, 0.56, 0.66, 0.49, 0.50],         # rank correlations of the
                  [0.29, 1.00, 0.40, 0.49, 0.17, 0.18, 0.14, 0.17],         # original study (rounded)
                  [0.19, 0.40, 1.00, 0.34, 0.13, 0.15, 0.25, 0.27],
                  [0.01, 0.49, 0.34, 1.00, -0.09, 0.02, 0.11, 0.10],
                  [0.56, 0.17, 0.13, -0.09, 1.00, 0.62, 0.38, 0.48],
                  [0.66, 0.18, 0.15, 0.02, 0.62, 1.00, 0.58, 0.66],
                  [0.49, 0.14, 0.25, 0.11, 0.38, 0.58, 1.00, 0.87],
                  [0.50, 0.17, 0.27, 0.10, 0.48, 0.66, 0.87, 1.00]])
    w, V = np.linalg.eigh(R)
    R = V @ np.diag(np.clip(w, 0.02, None)) @ V.T                        # make it positive definite
    Z = rng.multivariate_normal(np.zeros(8), R, n)
    from scipy.stats import norm
    U = norm.cdf(Z)
    floor = -1.38629
    df = pd.DataFrame({
        "lcavol": 1.35 + 1.18 * Z[:, 0],
        "lweight": 3.63 + 0.43 * Z[:, 1],
        "age": np.clip(np.round(63.9 + 7.4 * Z[:, 2]), 41, 79).astype(int),
        "lbph": np.where(U[:, 3] < 0.44, floor, np.clip(1.28 + 0.77 * norm.ppf((U[:, 3] - 0.44) / 0.56), -0.8, 2.4)),
        "svi": (U[:, 4] > 0.78).astype(int),
        "lcp": np.where(U[:, 5] < 0.46, floor, np.clip(0.87 + 1.13 * norm.ppf((U[:, 5] - 0.46) / 0.54), -0.8, 2.9)),
        "gleason": np.searchsorted([0.36, 0.94, 0.95], U[:, 6]) + 6,
    })
    df["pgg45"] = np.where(df["gleason"] == 6, 0, np.clip(5 * np.round((38 + 27 * Z[:, 7]) / 5), 5, 100)).astype(int)
    zs = (df - df.mean()) / df.std()
    coef = dict(lcavol=0.68, lweight=0.26, age=-0.14, lbph=0.21, svi=0.31, lcp=-0.29, gleason=-0.02, pgg45=0.27)
    df["lpsa"] = 2.47 + sum(c * zs[k] for k, c in coef.items()) + rng.normal(0, 0.71, n)
    for col in ("lcavol", "lweight", "lbph", "lcp", "lpsa"):
        df[col] = np.round(df[col], 5)
    df["train"] = rng.permutation(np.r_[np.ones(67, bool), np.zeros(30, bool)])
    return df[names + ["lpsa", "train"]]


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def logsumexp(a, axis=None):
    """log(Σ exp(a)) computed stably (subtract the maximum first)."""
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True)) + m
    return np.squeeze(out, axis=axis) if axis is not None else out.item()


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'simulate_prostate_clinical', 'sigmoid', 'logsumexp', 'standardize']))

In [ ]:
# ── 0.1 Generate the data used in this module (SIMULATED, fixed default seeds) ──
prostate = simulate_prostate_clinical()
FEATURES = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
leuk = simulate_leukemia_expression()
print("prostate: SIMULATED clinical data (shaped like Stamey et al. 1989)", prostate.shape)
print("leukaemia:", leuk["source"], leuk["X"].shape)
GENE_INDEX = {g: i for i, g in enumerate(leuk["genes"])}          # gene symbol → column

---
## 1 · Derivatives: slopes and rates of change

> 📍 **Where we are.** Modules 01–03 gave you arrays, probability and matrices — the language for *writing down* a
> model. To *fit* a model we need to know how the score changes when we nudge a parameter. That rate of change is the
> **derivative**, and it powers every learning algorithm in the course.

🧮 **Math refresher — the derivative.** For a function $f(x)$, the slope of the straight line through the points
$(x, f(x))$ and $(x+h, f(x+h))$ is $\frac{f(x+h) - f(x)}{h}$ (rise over run). Making the step $h$ smaller and smaller
gives the slope of the curve *at* $x$:

$$f'(x) = \frac{df}{dx}(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}.$$

*In words:* the derivative is the instantaneous rate of change — how much $f$ goes up per unit increase of $x$, right
at the point $x$. Positive: $f$ is increasing; negative: decreasing; zero: flat (a candidate top or bottom). "$\lim$"
(limit) means "the value the expression approaches as $h$ gets closer and closer to 0".

A biological example: a bacterial culture that doubles every 20 minutes, $N(t) = N_0\,2^{t/20}$. Its derivative
$N'(t)$ is the **growth rate** in cells per minute.

In [ ]:
# ── 1.1 Secant slopes approach the tangent slope: growth rate of a SIMULATED E. coli culture ──
N0 = 1000.0                                           # cells at t = 0

def culture(t):
    """Number of cells after t minutes for a doubling time of 20 min."""
    return N0 * 2 ** (t / 20)


t0 = 60.0
exact = np.log(2) / 20 * culture(t0)                  # derivative of N0·2^(t/20) = (ln 2 / 20)·N(t)
for h in [20, 10, 1, 0.01, 1e-6]:
    print(f"h = {h:>8g} min: secant slope = {(culture(t0 + h) - culture(t0)) / h:9.3f} cells/min")
print(f"tangent slope (calculus): (ln 2 / 20) · N(60) = {exact:.3f} cells/min")

ts = np.linspace(0, 100, 200)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(ts, culture(ts), color="tab:blue", label="N(t) = 1000 · 2^(t/20)")
for h, c in [(20, "tab:orange"), (10, "tab:green")]:
    slope = (culture(t0 + h) - culture(t0)) / h
    ax.plot([t0 - 15, t0 + h + 5], culture(t0) + slope * (np.array([t0 - 15, t0 + h + 5]) - t0), "--", color=c,
            lw=1, label=f"secant, h = {h} min (slope {slope:.0f})")
ax.plot(ts[60:140], culture(t0) + exact * (ts[60:140] - t0), color="tab:red", lw=2, label=f"tangent (slope {exact:.0f})")
ax.plot(t0, culture(t0), "ko")
ax.set_xlabel("time (min)"); ax.set_ylabel("number of cells"); ax.set_ylim(0, 30000)
ax.set_title("SIMULATED culture: the derivative is the growth rate at t = 60 min"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At $t = 60$ min the culture has 8,000 cells. The slope of the line to the point 20 minutes
later is 400 cells/min, to the point 10 minutes later 331, to the point 1 minute later 282 — and as $h \to 0$ the
slopes settle at **277.3 cells/min**, exactly the calculus answer $\frac{\ln 2}{20}\times 8000$. The red tangent line
just touches the curve. Notice that the growth rate is proportional to the number of cells: for exponential growth the
derivative of $N$ is a constant times $N$ itself — the reason $e^x$ is its own derivative.

### 1.2 The rules — and a numerical check for every one

You never need the limit definition in practice; a handful of rules gives every derivative in this course:

| Rule | Function | Derivative |
|---|---|---|
| constant | $c$ | $0$ |
| power | $x^n$ | $n\,x^{n-1}$ |
| constant multiple, sum | $a f(x) + b g(x)$ | $a f'(x) + b g'(x)$ |
| exponential | $e^x$ | $e^x$ |
| natural logarithm | $\ln x = \log x$ | $1/x$ |
| product | $f(x)\,g(x)$ | $f'(x)\,g(x) + f(x)\,g'(x)$ |
| **chain** | $f(g(x))$ | $f'(g(x))\cdot g'(x)$ |

The **chain rule** ("derivative of the outside, evaluated at the inside, times the derivative of the inside") is the one
that matters most: backpropagation in neural networks (Module 22) is the chain rule applied many times. The two special
cases that the logistic-regression lecture uses to differentiate its log-likelihood are $(\log f)' = f'/f$ and
$(e^f)' = e^f f'$.

To check a derivative without trusting our algebra, we use the **central finite difference**

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} \qquad (h \text{ small, e.g. } 10^{-5}).$$

*In words:* the slope of the line through two points just left and right of $x$; it is much more accurate than the
one-sided version for the same $h$.

In [ ]:
# ── 1.2 Check the rules numerically ──
def numerical_derivative(f, x, h=1e-5):
    """Central finite difference (f(x+h) − f(x−h)) / (2h): an approximation of f'(x)."""
    return (f(x + h) - f(x - h)) / (2 * h)


checks = [
    ("power      x^3",          lambda x: x ** 3,                   lambda x: 3 * x ** 2),
    ("exp+chain  e^(2x)",       lambda x: np.exp(2 * x),            lambda x: 2 * np.exp(2 * x)),
    ("log        ln x",         np.log,                             lambda x: 1 / x),
    ("product    x^2 ln x",     lambda x: x ** 2 * np.log(x),       lambda x: 2 * x * np.log(x) + x),
    ("chain      (3x − 1)^4",   lambda x: (3 * x - 1) ** 4,         lambda x: 12 * (3 * x - 1) ** 3),
    ("chain      ln(1 + e^x)",  lambda x: np.log1p(np.exp(x)),      lambda x: np.exp(x) / (1 + np.exp(x))),
]
x0 = 1.7
rows = [dict(rule=name, analytic=d(x0), numerical=numerical_derivative(f, x0)) for name, f, d in checks]
tab = pd.DataFrame(rows); tab["abs. difference"] = (tab.analytic - tab.numerical).abs()
print(f"derivatives at x = {x0}:"); print(tab.to_string(index=False, float_format=lambda v: f"{v:.8g}"))

print("\n⚠️ the step h matters (derivative of e^x at x = 1, true value e = 2.718281828…):")
for h in [1e-1, 1e-3, 1e-5, 1e-8, 1e-11, 1e-13]:
    print(f"   h = {h:.0e}: error = {abs(numerical_derivative(np.exp, 1.0, h) - np.e):.1e}")

🔍 **What just happened?** For every rule the analytic derivative and the finite-difference estimate agree to at least
seven significant digits (absolute differences of at most $5\times10^{-8}$, even for a derivative as large as 827). The last function, $\ln(1+e^x)$ (called "softplus"), appears in the logistic-regression
log-likelihood (Section 5.2); its derivative $e^x/(1+e^x)$ is the logistic function itself — keep that in mind.

⚠️ **Pitfall.** Smaller $h$ is *not* always better. The error first shrinks (the approximation improves) and then grows
again (from about $6\times10^{-11}$ at $h = 10^{-5}$ to $10^{-5}$ at $h = 10^{-11}$), because the computer stores numbers with only about 16 significant digits and
$f(x+h) - f(x-h)$ becomes a difference of two almost equal numbers. $h \approx 10^{-5}$ is a good default for a central
difference.

### 1.3 The derivative of the logistic function

The logistic (sigmoid) function turns any number into a probability and is the heart of logistic regression
(Module 09), where $P(Y = 1\mid X, W) = \sigma(w_0 + \sum_i w_i X_i)$, and of neural networks:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \sigma(z)\,\big(1 - \sigma(z)\big).$$

*In words:* the slope of the S-curve at any point equals "probability × (1 − probability)": steepest (0.25) in the
middle where $\sigma = 0.5$, nearly flat in the tails where the model is already confident.

**Derivation with the chain rule.** Write $\sigma(z) = (1+e^{-z})^{-1}$. Outside function $u^{-1}$ (derivative $-u^{-2}$),
inside $u = 1 + e^{-z}$ (derivative $-e^{-z}$, chain rule again). So

$$\sigma'(z) = -\big(1+e^{-z}\big)^{-2}\cdot\big(-e^{-z}\big) = \frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}} = \sigma(z)\,\big(1-\sigma(z)\big),$$

*In words:* the second fraction is $1 - \sigma(z)$ because $1 - \frac{1}{1+e^{-z}} = \frac{e^{-z}}{1+e^{-z}}$.

In [ ]:
# ── 1.3 σ'(z) = σ(z)(1 − σ(z)), checked numerically (using the toolkit's sigmoid) ──
for z in [-4.0, -1.0, 0.0, 0.5, 3.0]:
    s = sigmoid(z)
    print(f"z = {z:+.1f}: σ = {s:.4f}   σ(1−σ) = {s * (1 - s):.6f}   finite difference = {numerical_derivative(sigmoid, z):.6f}")

zs = np.linspace(-7, 7, 300)
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(zs, sigmoid(zs), color="tab:blue", label="σ(z): e.g. P(AML | expression score z)")
ax.plot(zs, sigmoid(zs) * (1 - sigmoid(zs)), color="tab:orange", label="σ'(z) = σ(z)(1 − σ(z))")
ax.axhline(0.25, color="0.6", ls=":", lw=1); ax.axvline(0, color="0.6", ls=":", lw=1)
ax.set_xlabel("z = w0 + w1·x  (linear score)"); ax.set_ylabel("value"); ax.legend(fontsize=7)
ax.set_title("the logistic curve is steepest (slope 0.25) where σ = 0.5"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The formula $\sigma(1-\sigma)$ matches the finite difference to six decimals at every $z$,
and the orange curve peaks at 0.25 at $z=0$. Two consequences you will meet again: the gradient of the logistic
log-likelihood contains the simple factor $y - \sigma(\cdot)$ (Section 5.2), and in
deep networks, many multiplied factors of at most 0.25 make gradients shrink layer after layer — the "vanishing
gradient" problem of Module 22.

✏️ **Your turn.** Use the chain rule to differentiate $f(z) = \ln \sigma(z)$ and check your answer with
`numerical_derivative`. (Answer: $1 - \sigma(z)$.) This is the derivative of the log-probability of a positive example in
logistic regression.

---
## 2 · Finding the best value by setting the derivative to zero

> 📍 **Where we are.** At the top of a hill or the bottom of a valley the slope is zero. So "find the best parameter"
> becomes "solve $f'(\theta) = 0$" — the maximum-likelihood recipe of the 02-620 introduction: if the log-likelihood is
> differentiable, differentiate it with respect to θ and set the derivative to 0. We try it on the simplest learning
> problem there is.

🧫 You measured the expression of a gene in 7 replicate wells and want **one number** to report. Which number $m$
best summarises $x_1, \dots, x_7$? It depends on how you score a mistake.

**Squared error.** Score $m$ by the total squared distance to the data and minimise it:

$$L_2(m) = \sum_{i=1}^{N} (x_i - m)^2, \qquad L_2'(m) = -2\sum_{i=1}^{N} (x_i - m) = 0 \;\Longrightarrow\; m^\ast = \frac{1}{N}\sum_{i=1}^N x_i = \bar x .$$

*In words:* differentiate each squared term with the chain rule (outside $u^2$, inside $x_i - m$ with derivative $-1$),
set the sum to zero, and solve: the best summary under squared error is the **mean**. The second derivative $L_2''(m) = 2N > 0$ says the curve bends upward, so this flat point is a minimum, not a maximum.

**Absolute error.** Score by $L_1(m) = \sum_i |x_i - m|$. The derivative of $|x_i - m|$ is $-1$ if $m < x_i$ and $+1$ if
$m > x_i$, so

$$L_1'(m) = \#\{i: x_i < m\} - \#\{i: x_i > m\} = 0 \;\Longrightarrow\; m^\ast = \text{median}.$$

*In words:* the slope counts how many points lie below minus how many lie above $m$; it is zero when half the points are
on each side — the **median**.

In [ ]:
# ── 2.1 Mean vs median: the minimisers of squared and absolute error (SIMULATED replicate wells) ──
wells = np.array([5.1, 4.8, 5.4, 5.0, 4.9, 5.3, 5.2])          # expression in 7 replicate wells (a.u.)
wells_bad = wells.copy(); wells_bad[-1] = 9.8                     # the same, with one failed/contaminated well
grid = np.linspace(4.0, 10.5, 1301)

def sq_loss(m, x):
    """Total squared error of the summary m."""
    return np.sum((x[None, :] - np.atleast_1d(m)[:, None]) ** 2, axis=1)

def abs_loss(m, x):
    """Total absolute error of the summary m."""
    return np.sum(np.abs(x[None, :] - np.atleast_1d(m)[:, None]), axis=1)


for name, x in [("clean", wells), ("with outlier", wells_bad)]:
    print(f"{name:13s}: argmin squared = {grid[sq_loss(grid, x).argmin()]:.3f} (mean {x.mean():.3f}) | "
          f"argmin absolute = {grid[abs_loss(grid, x).argmin()]:.3f} (median {np.median(x):.3f})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
for ax, loss, title in [(axes[0], sq_loss, "squared error Σ(xᵢ − m)²"), (axes[1], abs_loss, "absolute error Σ|xᵢ − m|")]:
    for x, c, lab in [(wells, "tab:blue", "clean"), (wells_bad, "tab:red", "one outlier (9.8)")]:
        L = loss(grid, x); ax.plot(grid, L, color=c, label=lab); ax.plot(grid[L.argmin()], L.min(), "o", color=c)
    ax.set_xlabel("candidate summary m"); ax.set_ylabel("total error"); ax.set_title(title); ax.legend(fontsize=7)
axes[0].set_ylim(0, 40); axes[1].set_ylim(0, 15)
for ax in axes:
    ax.set_xlim(4, 8)
    ax.plot(wells_bad, np.zeros_like(wells_bad), "|", color="k", ms=10)   # the data (the outlier at 9.8 is off-screen)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The grid search lands exactly on the formulas: the squared-error minimum is the mean (5.10 for
the clean wells) and the absolute-error minimum is the median (5.10). One contaminated well at 9.8 drags the mean to
5.76 but leaves the median at 5.10: squared error punishes the outlier's huge residual so heavily that the summary moves
toward it. The squared-error curve is a smooth parabola (easy to differentiate); the absolute-error curve is made of
straight pieces with kinks at the data points (its "derivative" jumps — the reason the lasso of Module 07 needs special
treatment).

⚠️ **Why this matters for 02-620.** Minimising squared error is what linear regression does (Module 06), and Module 05
shows it is the same as maximising a **Gaussian** likelihood. Its sensitivity to outliers
is inherited by every least-squares method; robust alternatives (absolute error ↔ a Laplace noise model) trade a smooth
problem for robustness.

✏️ **Your turn.** Minimise the *weighted* squared error $\sum_i v_i (x_i - m)^2$ (e.g. weights $v_i$ = number of cells
per well). Set the derivative to zero by hand, then confirm on a grid with `v = np.array([1, 1, 1, 1, 1, 1, 5])`.

---
## 3 · Logarithms turn products into sums

> 📍 **Where we are.** A likelihood is a *product* of one probability per observation (Module 05). Section 2 showed
> that we optimise by differentiating — and products of thousands of terms are painful to differentiate and, as we
> will see, impossible even to store. Logarithms fix both problems, which is why every likelihood in the course is
> maximised in the form $\log\prod$.

🧮 **Math refresher — logarithms.** $\log$ means the natural logarithm ($\ln$, base $e \approx 2.718$) throughout the
course. Three rules:

$$\log(ab) = \log a + \log b, \qquad \log a^k = k\log a, \qquad \log\prod_{i=1}^{N} p_i = \sum_{i=1}^{N}\log p_i .$$

*In words:* the log of a product is the sum of the logs ($\prod$ is the product symbol, the multiplication analogue of
$\sum$). Because $\log$ is strictly increasing, the parameter that maximises a likelihood also maximises its log — so we
may always work with the friendlier **log-likelihood**.

In [ ]:
# ── 3.1 The likelihood of 1,000 genotypes underflows; the log-likelihood does not (SIMULATED) ──
rng = np.random.default_rng(SEED + 3)
g = rng.binomial(2, 0.3, size=1000)                      # 1,000 genotypes at a SNP with allele frequency 0.3

def genotype_probs(g, p):
    """Hardy–Weinberg probability of each genotype (0, 1, 2 copies) for allele frequency p."""
    return np.array([(1 - p) ** 2, 2 * p * (1 - p), p ** 2])[g]


for p in [0.30, 0.35]:
    probs = genotype_probs(g, p)
    print(f"p = {p}: product of 1,000 probabilities = {np.prod(probs)}   sum of logs = {np.sum(np.log(probs)):.2f}")
cum = np.cumprod(genotype_probs(g, 0.3))
first_zero = int(np.argmax(cum == 0))
print(f"the running product becomes exactly 0.0 after {first_zero} genotypes; smallest positive float64 ≈ {np.nextafter(0, 1):.1e}")

a = np.log(genotype_probs(g, 0.3)).sum() + np.log([0.5, 0.5])   # log of ½·L(0.30) and of ½·L(0.30) again
with np.errstate(divide="ignore"):                       # the naive version is SUPPOSED to fail here
    naive = np.log(np.sum(np.exp(a)))
print(f"log(½L + ½L): naive = {naive}   logsumexp = {logsumexp(a):.2f}")

fig, ax = plt.subplots(figsize=(6.4, 3))
ax.plot(np.cumsum(np.log10(genotype_probs(g, 0.3))), color="tab:blue", label="log10 of the running product (via a sum of logs)")
ax.axhline(np.log10(np.nextafter(0, 1)), color="tab:red", ls="--", lw=1, label="smallest positive float64 (10^−323.3)")
ax.axvline(first_zero, color="0.5", ls=":", lw=1)
ax.set_xlabel("number of genotypes multiplied"); ax.set_ylabel("log10 likelihood"); ax.legend(fontsize=7)
ax.set_title("SIMULATED genotypes: the raw product underflows to 0"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Each genotype has probability 0.09–0.49, so the likelihood of 1,000 genotypes is around
$10^{-400}$ — smaller than the smallest number a computer can store (about $10^{-323}$). The product therefore
**underflows to exactly 0.0** (here after 823 genotypes), and both allele frequencies get "likelihood 0": we can
no longer tell which fits better. The sums of logs are ordinary numbers and clearly prefer $p = 0.30$ (the true value).

Sometimes we need the log of a **sum** of tiny likelihoods (mixtures, Module 13; HMMs, Module 19). Exponentiating first
underflows again and gives $\log 0 = -\infty$; the toolkit's `logsumexp` subtracts the largest term first,
$\log\sum_k e^{a_k} = m + \log\sum_k e^{a_k - m}$ with $m = \max_k a_k$, and returns the right answer.

✏️ **Your turn.** Compute the log-likelihood of the 1,000 genotypes for $p$ on a grid from 0.01 to 0.99 and find the
maximiser. Compare it with the allele-counting estimate `g.sum() / (2 * len(g))` — Module 05 proves they are equal.

---
## 4 · Gradients and gradient descent

> 📍 **Where we are.** For the mean we could solve $f'(m) = 0$ on paper. For logistic regression, neural networks and
> mixtures we cannot — the logistic-regression lecture puts it bluntly: "bad news: no closed-form solution". Then we do
> what the hiker in the fog does: take a step in the downhill (or uphill) direction, re-measure the slope, repeat.

**Gradient ascent** (to maximise $f$) and **gradient descent** (to minimise) in one dimension. To find the $x$ that
maximises $f(x)$: start from a random initial value $x_0$, and for $t = 0, 1, 2, \dots$

$$x_{t+1} = x_t + \lambda\, f'(x_t) \quad\text{(ascent)}, \qquad x_{t+1} = x_t - \lambda\, f'(x_t) \quad\text{(descent)} .$$

*In words:* from the current guess $x_t$, move in the direction in which $f$ increases (or decreases), by a distance
proportional to the slope; the **step size** $\lambda$ (also written $\eta$, the "learning rate") controls how bold
each step is. Stop when the improvement $|f(x_{t-1}) - f(x_t)|$ falls below a small tolerance $\varepsilon$. The
lecture's two warnings: $\lambda$ too large → **overshoot**; $\lambda$ too small → **too many iterations**.

⚠️ **Notation alert.** The letter $\lambda$ is badly overloaded in 02-620: a step size here, a Lagrange multiplier and
an eigenvalue in PCA (Section 7), and a regularisation strength in ridge regression (Module 07). Always check which one
a formula means.

We maximise $f(m) = -\tfrac12\sum_i (x_i - m)^2$ for the 7 clean wells of Section 2 (whose maximiser we know: the mean,
5.1). Its slope is $f'(m) = \sum_i (x_i - m) = N(\bar x - m)$.

In [ ]:
# ── 4.1 Gradient ascent in 1-D: too small, good, overshooting and diverging step sizes ──
def f_wells(m):
    """Objective to MAXIMISE: −½ Σ (x_i − m)² for the 7 clean wells (maximum at the mean)."""
    return -0.5 * np.sum((wells - m) ** 2)

def df_wells(m):
    """Its derivative N (x̄ − m)."""
    return np.sum(wells - m)

def gradient_ascent_1d(df, x0, step, n_steps=40, f=None, tol=None):
    """x_{t+1} = x_t + step · f'(x_t). Stops early when |f(x_{t-1}) − f(x_t)| < tol (if f and tol are given)."""
    xs = [x0]
    for _ in range(n_steps):
        xs.append(xs[-1] + step * df(xs[-1]))
        if tol is not None and abs(f(xs[-1]) - f(xs[-2])) < tol:
            break
    return np.array(xs)


steps = {0.02: "tab:blue", 0.12: "tab:green", 0.27: "tab:orange", 0.30: "tab:red"}
paths = {s: gradient_ascent_1d(df_wells, x0=3.0, step=s) for s in steps}
for s, xs in paths.items():
    print(f"step λ = {s:.2f} (λN = {s * len(wells):.2f}): x after 5 steps = {xs[5]:8.3f}, after 40 = {xs[-1]:10.3f}")
xs_tol = gradient_ascent_1d(df_wells, 3.0, 0.12, n_steps=1000, f=f_wells, tol=1e-10)
print(f"with the stopping rule |Δf| < 1e-10 and λ = 0.12: stopped after {len(xs_tol) - 1} steps at m = {xs_tol[-1]:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
mm = np.linspace(2.5, 7.5, 300)
axes[0].plot(mm, [f_wells(m) for m in mm], color="0.6")
for s, xs in paths.items():
    if s < 0.3:
        axes[0].plot(xs[:8], [f_wells(m) for m in xs[:8]], "o-", ms=3, lw=1, color=steps[s], label=f"λ = {s}")
axes[0].set_ylim(-25, 1); axes[0].set_xlabel("m"); axes[0].set_ylabel("f(m)"); axes[0].legend(fontsize=7)
axes[0].set_title("first 7 steps from m₀ = 3")
for s, xs in paths.items():
    axes[1].semilogy(np.abs(xs - wells.mean()) + 1e-16, "o-", ms=2, lw=1, color=steps[s], label=f"λ = {s}")
axes[1].set_xlabel("iteration t"); axes[1].set_ylabel("|m_t − x̄|  (log scale)"); axes[1].set_ylim(1e-12, 1e3)
axes[1].set_title("λ↓: too many iterations; λ↑: overshoot, divergence"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** All four runs start at $m_0 = 3$. With $\lambda = 0.02$ the hiker creeps: after 40 steps she
is still short of 5.1 (the blue line in the right panel descends slowly). With $\lambda = 0.12$ she arrives in a few steps
and the stopping rule $|\Delta f| < 10^{-10}$ ends the run after only 9 steps. With $\lambda = 0.27$ every step **overshoots**
the top and lands on the other side (zig-zag in the left panel) but still converges, slowly. With $\lambda = 0.30$ each
overshoot is bigger than the last and the iterates **diverge**. These are exactly the lecture's two warnings:
"λ ↑ : overshoot, λ ↓ : too many iterations".

Why these thresholds? Here $m_{t+1} - \bar x = (1 - \lambda N)(m_t - \bar x)$: each step multiplies the error by
$(1 - \lambda N)$. It shrinks steadily if $0 < \lambda N < 1$, shrinks while flipping sign if $1 < \lambda N < 2$, and
grows if $\lambda N > 2$ ($N = 7$, so the critical step size is $2/7 \approx 0.286$). In general the safe step size is
set by the **curvature** (second derivative) of the objective.

### 4.2 Several parameters: partial derivatives and the gradient

🧮 **Math refresher — partial derivatives.** For a function of several variables, $f(w_0, w_1)$, the **partial
derivative** $\partial f/\partial w_1$ is the ordinary derivative with respect to $w_1$ while $w_0$ is held fixed. The
**gradient** collects all partial derivatives in a vector:

$$\nabla f(w_0, w_1) = \begin{pmatrix} \partial f / \partial w_0 \\ \partial f / \partial w_1 \end{pmatrix} .$$

*In words:* the gradient points in the direction of steepest **ascent**, its length is the steepness, and it is
perpendicular to the **contour lines** (lines of equal $f$, like the height lines of a hiking map). On a contour plot,
gradient ascent from $\mathbf x_0$ climbs through $\mathbf x_1, \mathbf x_2, \dots$ toward the summit, each step crossing
the contours at a right angle. Gradient descent steps along $-\nabla f$.

On the prostate data: predict log PSA from log cancer volume with a line, $\hat y = w_0 + w_1 x$, and score the line by the mean
squared error $E(w_0,w_1) = \frac1N\sum_i (y_i - w_0 - w_1 x_i)^2$. By the chain rule,

$$\frac{\partial E}{\partial w_0} = -\frac{2}{N}\sum_{i=1}^{N}\big(y_i - w_0 - w_1 x_i\big), \qquad
\frac{\partial E}{\partial w_1} = -\frac{2}{N}\sum_{i=1}^{N} x_i\big(y_i - w_0 - w_1 x_i\big).$$

*In words:* both partial derivatives are (minus twice) averages of the residuals — the second one weighted by $x_i$.
At the best line both are zero: the residuals average to zero and are uncorrelated with $x$.

In [ ]:
# ── 4.2 The loss surface of a one-predictor regression on the prostate data ──
x_pr = prostate["lcavol"].to_numpy(float)
y_pr = prostate["lpsa"].to_numpy(float)

def mse(w0, w1, x=x_pr, y=y_pr):
    """Mean squared error of the line w0 + w1·x (works on grids of w0, w1 too)."""
    w0, w1 = np.asarray(w0, float), np.asarray(w1, float)
    return np.mean((y - w0[..., None] - w1[..., None] * x) ** 2, axis=-1)

def mse_grad(w, x=x_pr, y=y_pr):
    """Gradient (∂E/∂w0, ∂E/∂w1) from the formulas above."""
    r = y - w[0] - w[1] * x
    return np.array([-2 * r.mean(), -2 * (x * r).mean()])

def numerical_gradient(f, w, h=1e-5):
    """Central finite-difference gradient of a scalar function f at the vector w."""
    w = np.asarray(w, float); grad = np.zeros_like(w)
    for j in range(len(w)):
        e = np.zeros_like(w); e[j] = h
        grad[j] = (f(w + e) - f(w - e)) / (2 * h)
    return grad


w_test = np.array([0.5, 0.2])
print("gradient at (0.5, 0.2): formula", mse_grad(w_test).round(6), "| finite differences",
      numerical_gradient(lambda w: float(mse(w[0], w[1])), w_test).round(6))
w_best = np.polyfit(x_pr, y_pr, 1)[::-1]                    # least-squares line (intercept, slope)
print("best line (np.polyfit): w0 = %.4f, w1 = %.4f; gradient there:" % tuple(w_best), mse_grad(w_best).round(10) + 0.0)

🔍 **What just happened?** The gradient formula agrees with the finite-difference gradient, and at the least-squares line
($w_0 \approx 1.47$, $w_1 \approx 0.70$: each unit of log cancer volume adds about 0.70 to log PSA) the gradient is zero
to ten decimals — the flat bottom of the valley. Now let gradient descent find that bottom by itself.

In [ ]:
# ── 4.3 Gradient-descent paths on the contour plot: raw vs centred predictor ──
def gradient_descent(grad, w0, step, n_steps=500, f=None, tol=1e-10):
    """w ← w − step · ∇f(w); stops when |f(w_{t−1}) − f(w_t)| < tol. Returns the path (one row per iterate)."""
    path = [np.asarray(w0, float)]
    for _ in range(n_steps):
        path.append(path[-1] - step * grad(path[-1]))
        if f is not None and abs(f(path[-1]) - f(path[-2])) < tol:
            break
    return np.array(path)


x_cen = x_pr - x_pr.mean()                                   # centred predictor (same slope, new intercept)
cases = {"raw lcavol": x_pr, "centred lcavol": x_cen}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, (name, xv) in zip(axes, cases.items()):
    f2 = lambda w, xv=xv: float(mse(w[0], w[1], x=xv))
    g2 = lambda w, xv=xv: mse_grad(w, x=xv)
    path = gradient_descent(g2, [-1.0, -0.5], step=0.2, f=f2)
    W0, W1 = np.meshgrid(np.linspace(-1.5, 4, 120), np.linspace(-0.8, 1.8, 120))
    ax.contour(W0, W1, mse(W0, W1, x=xv), levels=np.geomspace(0.5, 30, 14), cmap="viridis", linewidths=0.8)
    ax.plot(path[:, 0], path[:, 1], "o-", ms=2, lw=1, color="tab:red")
    ax.plot(*path[-1], "k*", ms=12)
    ax.set_xlabel("intercept w0"); ax.set_ylabel("slope w1")
    ax.set_title(f"{name}: {len(path) - 1} steps (λ = 0.2) → w = ({path[-1][0]:.2f}, {path[-1][1]:.2f})")
    print(f"{name:15s}: converged after {len(path) - 1} steps to w0 = {path[-1][0]:.4f}, w1 = {path[-1][1]:.4f}")
fig.suptitle("SIMULATED prostate data: mean squared error of the line lpsa ≈ w0 + w1·lcavol", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the raw predictor the valley is a long, tilted, narrow ellipse: intercept and slope are
entangled (changing the slope moves the line far away from the data unless the intercept compensates, because
`lcavol` values are far from zero). The path first bounces back and forth across the narrow valley — the step size is
close to the stability limit in the steep direction — and then crawls along the flat floor: 62 steps in total. After **centring** the predictor, the contours are axis-aligned ellipses, the two parameters no longer interfere,
and the same step size converges in 26 steps to the same slope 0.70 (the
intercept changes meaning: it is now the predicted log PSA at the *average* cancer volume, 2.38). The shape of the valley
is described by the eigenvalues of the matrix of second derivatives (the **Hessian**, Section 5): long thin valleys
have one large and one small eigenvalue. This is why Modules 06–09 and 22 centre or standardise features before
optimising.

✏️ **Your turn.** Rerun 4.3 with step sizes 0.05 and 0.3 for the raw predictor. Which one is too slow and which one
diverges? (Hint: the largest eigenvalue of the Hessian of $E$ is about 7.8, so steps above $2/7.78 \approx 0.257$ diverge.)

### 📘 Lecture notes — gradient ascent (the training algorithm of logistic regression)

**The algorithm.** To find the $x$ that maximises $f(x)$: start from a random $x_0$; repeat
$x_{t+1} = x_t + \lambda f'(x_t)$ with step size $\lambda$ until the change is small. For a model with parameters
$W = (w_0, \dots, w_n)$ and log-likelihood $l(W)$:

$$\text{repeat for all } i:\quad w_i \leftarrow w_i + \eta\,\frac{\partial l(W)}{\partial w_i}
\qquad\text{until}\quad \big|l(W^{(t-1)}) - l(W^{(t)})\big| < \varepsilon .$$

*In words:* nudge every weight in the direction that increases the log-likelihood, by an amount proportional to the
partial derivative; $\eta$ (or $\lambda$ — the same step size under two names) sets the boldness, $\varepsilon$ the
stopping tolerance. Descent on $-l$ is the same algorithm with the opposite sign.

**Choosing the step size.** Too large → overshoot (zig-zag across the optimum, or divergence); too small → too many
iterations. For $f(m) = -\frac12\sum_i (x_i - m)^2$ each step multiplies the error by $(1 - \lambda N)$: steady
convergence for $0 < \lambda N < 1$, oscillating convergence for $1 < \lambda N < 2$, divergence for $\lambda N > 2$
(Section 4.1, $N = 7$: critical step $2/7 \approx 0.29$). In general the safe step is below $2/$(largest eigenvalue of
the Hessian).

**Geometry.** The gradient $\nabla f$ points uphill, perpendicular to the contour lines; its length is the
steepness. Long, thin valleys (very different curvatures in different directions) make gradient methods zig-zag;
centring or standardising the features rounds the valley (Section 4.3).

> 🎓 **Exam tip.** Be able to write the update with the correct sign (ascent: $+$, descent: $-$), to name a stopping rule, and to
    explain what goes wrong for too large / too small a step. "Gradient ascent on $l$" and "gradient descent on $-l$" are
    the same algorithm.

---
## 5 · Convex vs non-convex: Mount Fuji or the Himalayas?

> 📍 **Where we are.** Gradient methods only feel the *local* slope. Whether the top they reach is *the* top depends on
> the shape of the landscape. Right after the bad news, the logistic-regression lecture announces the good news —
> "$l(W)$ is a **concave** function of $W$", so gradient ascent will reach the maximum — with a memorable picture:
> Mount Fuji (one summit) instead of the Himalayas (many peaks). Section 5.4 draws both landscapes.

🧮 **Math refresher — convexity.** A function is **convex** if every chord lies on or above the curve, and
**concave** if every chord lies on or below it:

$$\text{convex: } f\big(t x_1 + (1-t) x_2\big) \le t f(x_1) + (1-t) f(x_2), \qquad
\text{concave: } f\big(t x_1 + (1-t) x_2\big) \ge t f(x_1) + (1-t) f(x_2), \qquad \text{for all } t\in[0,1],\ x_1, x_2 .$$

*In words:* the point $t x_1 + (1-t)x_2$ walks along the segment between $x_1$ and $x_2$; for a convex ("bowl") function
the curve sags below the straight line joining $f(x_1)$ and $f(x_2)$, for a concave ("dome") function it bulges above.
A convex function has **no local minima other than the global minimum**, and a concave one no local maxima other
than the global maximum — so gradient descent (ascent) cannot get stuck. A practical test: $f''(x) \ge 0$ everywhere
means convex; for several variables, the **Hessian** (matrix of second partial derivatives) must have only
non-negative eigenvalues (Module 03).

In [ ]:
# ── 5.1 Chords above and below: convex, concave and neither ──
funcs = {"convex: f(x) = x²": (lambda x: x ** 2, -2, 2),
         "concave: f(x) = ln x": (np.log, 0.2, 4),
         "neither: f(x) = sin(3x) + x²/4": (lambda x: np.sin(3 * x) + x ** 2 / 4, -3, 3)}
rng = np.random.default_rng(SEED + 5)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (name, (f, lo, hi)) in zip(axes, funcs.items()):
    x1, x2 = rng.uniform(lo, hi, 2000), rng.uniform(lo, hi, 2000)
    t = rng.uniform(0, 1, 2000)
    lhs, rhs = f(t * x1 + (1 - t) * x2), t * f(x1) + (1 - t) * f(x2)
    print(f"{name:32s}: chord above curve in {np.mean(lhs <= rhs + 1e-12):6.1%} of 2,000 random tests, "
          f"below in {np.mean(lhs >= rhs - 1e-12):6.1%}")
    xs = np.linspace(lo, hi, 300); ax.plot(xs, f(xs), color="tab:blue")
    a_, b_ = lo + 0.15 * (hi - lo), lo + 0.8 * (hi - lo)
    ax.plot([a_, b_], [f(a_), f(b_)], "o--", color="tab:red", lw=1.2, ms=4)
    ax.set_title(name, fontsize=9); ax.set_xlabel("x")
plt.tight_layout(); plt.show()

X1c = np.column_stack([np.ones_like(x_pr), x_pr])
H_mse = 2 * X1c.T @ X1c / len(x_pr)                         # Hessian of the mean squared error of Section 4
print("\nHessian of the prostate MSE:", H_mse.round(3).tolist(), "→ eigenvalues", np.linalg.eigvalsh(H_mse).round(3),
      "(all ≥ 0 ⇒ convex)")

🔍 **What just happened?** For $x^2$ the chord is above the curve in 100 % of 2,000 random tests (convex); for $\ln x$ it
is below in 100 % (concave); for the wiggly function it is sometimes above and sometimes below (neither). The squared
error of Section 4 is convex: its Hessian $2X^\top X/N$ has non-negative eigenvalues (here 0.84 and 7.78 — the
large/small pair behind the long thin valley of the raw predictor in 4.3). Matrices of the form $X^\top X$ never have
negative eigenvalues, so **every least-squares problem is convex**: one valley, one bottom.

### 5.2 Logistic regression on one gene: Mount Fuji

**The conditional log-likelihood.** Logistic regression models $P(Y = 1\mid X, W) = \frac{\exp(w_0 + \sum_i w_iX_i)}{1 + \exp(w_0 + \sum_i w_iX_i)}$
and $P(Y = 0\mid X, W) = \frac{1}{1 + \exp(w_0 + \sum_i w_iX_i)}$. Training maximises the **conditional** log-likelihood
of the $L$ training pairs, $l(W) = \sum_l \ln P(Y^l\mid X^l, W)$ (the maximum conditional likelihood estimate, MCLE). A
sample with $Y^l = 1$ contributes $\ln P(Y^l = 1\mid X^l, W)$, one with $Y^l = 0$ contributes $\ln P(Y^l = 0\mid X^l, W)$;
both cases fit into one expression,

$$l(W) = \sum_l\Big[Y^l\ln P(Y^l = 1\mid X^l, W) + (1 - Y^l)\ln P(Y^l = 0\mid X^l, W)\Big]
= \sum_{l}\Big[\,Y^l\big(w_0 + \textstyle\sum_i w_i X_i^l\big) - \ln\!\big(1 + e^{\,w_0 + \sum_i w_i X_i^l}\big)\Big] .$$

*In words:* the factor $Y^l$ switches on the first log-probability for positive samples and $1 - Y^l$ the second for
negative ones. Simplifying uses $\ln P(Y = 1\mid\cdot) = z - \ln(1 + e^z)$ and $\ln P(Y = 0\mid\cdot) = \ln 1 - \ln(1 + e^z)$
with $z = w_0 + \sum_i w_iX_i$, so the $\ln(1 + e^z)$ term appears for every sample. Training samples are indexed by the
superscript $l$ and features by $i = 1, \dots, n$ (the logistic-regression lecture's $n$ is our $J$).

**The gradient.** With the chain rule and Section 1.2's softplus derivative, $\frac{\partial}{\partial z}\ln(1+e^z) = \sigma(z)$:

$$\frac{\partial l}{\partial w_i} = \sum_{l} X_i^l\,\Big(Y^l - \hat P\big(Y^l = 1 \mid X^l, W\big)\Big), \qquad X_0^l = 1 .$$

*In words:* each sample pushes the weights in proportion to its **prediction error** $Y - \hat P$ (times the feature
value): correctly and confidently predicted samples barely matter, surprising ones matter a lot. There is no
closed-form solution of $\partial l/\partial w_i = 0$, but $l(W)$ is concave — so gradient ascent works.

We fit it for labels $y^l \in\{0,1\}$ (0 = ALL, 1 = AML) and a single standardised gene $x^l$, MYB (c-Myb), so that
$W = (w_0, w_1)$ and we can draw the whole landscape.

In [ ]:
# ── 5.2 Gradient ascent on a concave logistic log-likelihood (SIMULATED leukaemia data, gene MYB) ──
y_gl = leuk["y"].astype(float)
x_myb, _, _ = standardize(leuk["X"][:, GENE_INDEX["MYB"]])
x_myb = x_myb.ravel()                                        # z-scored expression of c-myb (all 72 patients)

def loglik_lr(w, x=x_myb, y=y_gl):
    """Conditional log-likelihood l(w0, w1) (works on grids of w0, w1 stacked in w[0], w[1])."""
    z = np.asarray(w[0])[..., None] + np.asarray(w[1])[..., None] * x
    return np.sum(y * z - np.logaddexp(0, z), axis=-1)      # logaddexp(0, z) = ln(1 + e^z) without overflow

def grad_lr(w, x=x_myb, y=y_gl):
    """Gradient Σ x_i (y − σ(w0 + w1 x)), with x_0 = 1."""
    err = y - sigmoid(w[0] + w[1] * x)
    return np.array([err.sum(), (x * err).sum()])


w_chk = np.array([0.3, -0.7])
print("gradient: formula", grad_lr(w_chk).round(6), "| finite differences",
      numerical_gradient(lambda w: float(loglik_lr(w)), w_chk).round(6))
ends = []
for start in [(-3.0, 1.5), (2.0, 1.0), (1.5, -4.5), (-3.5, -4.0)]:
    path = gradient_descent(lambda w: -grad_lr(w), start, step=0.05, n_steps=5000, f=lambda w: -float(loglik_lr(w)))
    ends.append(path[-1]); print(f"start {start} → ({path[-1][0]:+.4f}, {path[-1][1]:+.4f}) after {len(path) - 1} steps")
opt = minimize(lambda w: -float(loglik_lr(w)), x0=[0, 0], jac=lambda w: -grad_lr(w))
print("scipy.optimize.minimize:", opt.x.round(4), "| max log-likelihood", round(-opt.fun, 4))

🔍 **What just happened?** The gradient formula matches the finite differences. Gradient ascent started from
four very different corners always ends at the same point, $(w_0, w_1) \approx (-1.66, -4.49)$, which SciPy's optimiser
confirms: one summit, reached from everywhere — Mount Fuji. The negative slope says higher c-myb expression makes AML
*less* likely: one standard deviation more MYB multiplies the odds of AML by $e^{-4.49} \approx 0.01$. (We used all 72
patients because the point here is the shape of the landscape, not a fair test of prediction; Module 09 trains on the
original 38 training patients and tests on 34.) Notice also that the runs needed 250–300 steps: near-separable classes
make the summit a long, flat plateau (left panel of Section 5.4), and the stopping rule ends the climb on the plateau.

🧪 **Ground-truth check.** The simulator built MYB as an ALL marker: its true mean difference AML − ALL is −0.60 log10
units (the 72 simulated patients show −0.53, about 1.7 standard deviations of the gene). A strongly negative weight is
exactly what the truth predicts.

### 5.3 A mixture of two cell populations: the Himalayas

Now a non-concave likelihood. Suppose a marker is expressed at a low level in one cell population and a high level in
another, and a sample contains 60 % of the first and 40 % of the second (SIMULATED data). Modelling each population as a
Gaussian with known spread $\sigma$ and unknown centres $\mu_1, \mu_2$ gives the mixture log-likelihood (Module 13)

$$\ell(\mu_1,\mu_2) = \sum_{i=1}^{N}\log\Big[\,0.6\,\mathcal N(x_i;\mu_1,\sigma^2) + 0.4\,\mathcal N(x_i;\mu_2,\sigma^2)\Big] .$$

*In words:* each cell's expression is explained either by population 1 (with probability 0.6) or by population 2
(probability 0.4); the log of a **sum** makes the landscape bumpy. We compute it stably with the toolkit's `logsumexp`.

In [ ]:
# ── 5.3 Two summits: random restarts on a mixture log-likelihood (SIMULATED cells) ──
rng = np.random.default_rng(SEED + 53)
SIG = 0.8
cells_x = np.r_[rng.normal(2.0, SIG, 120), rng.normal(5.0, SIG, 80)]        # 60 % low, 40 % high population

def loglik_mix(mu, x=cells_x):
    """Mixture log-likelihood ℓ(μ1, μ2) with weights 0.6/0.4 and known σ (grids allowed)."""
    mu1, mu2 = np.asarray(mu[0])[..., None], np.asarray(mu[1])[..., None]
    lg = lambda m: -0.5 * ((x - m) / SIG) ** 2 - np.log(SIG * np.sqrt(2 * np.pi))
    return np.sum(logsumexp(np.stack([np.log(0.6) + lg(mu1), np.log(0.4) + lg(mu2)]), axis=0), axis=-1)

def grad_mix(mu, x=cells_x):
    """∂ℓ/∂μ_k = Σ_i r_ik (x_i − μ_k)/σ², with r_ik = P(population k | x_i) (the 'responsibilities' of EM)."""
    a = np.stack([np.log(0.6) - 0.5 * ((x - mu[0]) / SIG) ** 2, np.log(0.4) - 0.5 * ((x - mu[1]) / SIG) ** 2])
    r = np.exp(a - logsumexp(a, axis=0))
    return np.array([(r[0] * (x - mu[0])).sum(), (r[1] * (x - mu[1])).sum()]) / SIG ** 2


starts = rng.uniform(0, 7, size=(20, 2))
finals = np.array([gradient_descent(lambda m: -grad_mix(m), s, step=0.005, n_steps=3000,
                                    f=lambda m: -float(loglik_mix(m)))[-1] for s in starts])
lls = np.array([float(loglik_mix(m)) for m in finals])
for summit in np.unique(lls.round(1)):
    k = np.isclose(lls.round(1), summit)
    print(f"summit at (μ1, μ2) ≈ ({finals[k][0, 0]:.2f}, {finals[k][0, 1]:.2f}): log-likelihood {summit:8.1f}, "
          f"reached from {k.sum():2d} of 20 random starts")
print("best of 20 restarts:", finals[lls.argmax()].round(2))

In [ ]:
# ── 5.4 Picture: Mount Fuji (logistic) vs the Himalayas (mixture) ──
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
W0, W1 = np.meshgrid(np.linspace(-4.5, 2.5, 150), np.linspace(-7, 2, 150))
L = loglik_lr([W0, W1])
axes[0].contourf(W0, W1, L, levels=np.linspace(L.max() - 120, L.max(), 25), cmap="viridis", extend="min")
for e in ends:
    axes[0].plot(*e, "r*", ms=10)
axes[0].set_xlabel("w0"); axes[0].set_ylabel("w1 (per SD of MYB)")
axes[0].set_title("logistic log-likelihood (SIMULATED leukaemia, MYB):\none summit — Mount Fuji")
M1, M2 = np.meshgrid(np.linspace(0, 7, 150), np.linspace(0, 7, 150))
Lm = loglik_mix([M1, M2])
axes[1].contourf(M1, M2, Lm, levels=np.linspace(Lm.max() - 150, Lm.max(), 25), cmap="viridis", extend="min")
axes[1].plot(starts[:, 0], starts[:, 1], "w.", ms=6, label="random starts")
good = lls >= lls.max() - 1e-6
axes[1].plot(finals[good, 0], finals[good, 1], "r*", ms=13, label="global summit")
axes[1].plot(finals[~good, 0], finals[~good, 1], "^", color="tab:orange", ms=9, label="local summit")
axes[1].set_xlabel("μ1 (population 1 centre)"); axes[1].set_ylabel("μ2 (population 2 centre)")
axes[1].set_title("SIMULATED mixture log-likelihood:\ntwo summits — the Himalayas"); axes[1].legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The logistic landscape (left) has a single dome; all four runs of 5.2 end on the same star.
The mixture landscape (right) has **two summits**: the global one near $(\mu_1,\mu_2) \approx (2, 5)$ — the truth — and a
lower, *local* one near $(5, 2)$ where the labels are swapped (it fits worse because it gives the 60 % weight to the
high population). Gradient ascent climbs whichever summit is uphill from its starting point: the printout shows how many
of the 20 random starts ended on each. A single run could report the wrong answer with complete numerical confidence.
The remedy used throughout the course is **random restarts**: run from many starting points and keep the best
log-likelihood — exactly the advice for k-means (Module 12), EM (Module 13) and neural networks (Module 22).

Notice also what the mixture gradient contains: $r_{ik}$, the probability that cell $i$ belongs to population $k$. The
EM algorithm of Module 13 is built from these "responsibilities".

✏️ **Your turn.** Change the mixture weights in `loglik_mix` and `grad_mix` from 0.6/0.4 to 0.5/0.5. What happens to the
heights of the two summits, and why does it no longer matter which one gradient ascent finds?

### 📘 Lecture notes — maximising the logistic-regression log-likelihood: Mount Fuji, not the Himalayas

**The objective.** With $P(Y = 1\mid X, W) = \frac{\exp(w_0 + \sum_i w_iX_i)}{1 + \exp(w_0 + \sum_i w_iX_i)}$, the
maximum conditional likelihood estimate maximises

$$l(W) = \sum_l \ln P(Y^l\mid X^l, W) = \sum_l\Big[Y^l\big(w_0 + \textstyle\sum_i w_iX_i^l\big) - \ln\big(1 + \exp(w_0 + \sum_i w_iX_i^l)\big)\Big],$$

*in words:* split the samples into those with $Y^l = 1$ (contribute $\ln P(Y = 1\mid\cdot)$) and $Y^l = 0$
(contribute $\ln P(Y = 0\mid\cdot)$) and combine them with the switch $Y^l$.

**Bad news:** no closed-form solution. **Good news:** $l(W)$ is a **concave** function of $W$:

$$\text{convex: } f(tx_1 + (1-t)x_2) \le t f(x_1) + (1-t) f(x_2), \qquad \text{concave: } f(tx_1 + (1-t)x_2) \ge t f(x_1) + (1-t) f(x_2)$$

for all $t\in[0,1]$ and all $x_1, x_2$ — *in words:* chords lie below a concave curve (a dome) and above a convex one
(a bowl). A concave function has no local maxima except the global one, so gradient ascent with a suitable step
size reaches the maximum likelihood estimate: **Mount Fuji instead of the Himalayas**.

**The gradient** (using $(\log f)' = f'/f$ and $(e^f)' = e^f f'$):

$$\frac{\partial l(W)}{\partial w_i} = \sum_l X_i^l\big(Y^l - \hat P(Y^l = 1\mid X^l, W)\big), \qquad X_0^l = 1,$$

*in words:* the feature times the prediction error, summed over samples; the update is
$w_i \leftarrow w_i + \eta\,\partial l/\partial w_i$.

**Test for convexity:** $f'' \ge 0$ (one variable) or a Hessian with only non-negative eigenvalues (several);
least squares always passes, since its Hessian $2X^\top X/N$ has no negative eigenvalues.

**Worked example (Sections 5.2–5.3).** One standardised gene, MYB: four gradient-ascent runs from very different
corners end on the same summit. A two-population mixture: two summits, and 20 random restarts are needed to find
the global one.

> 🎓 **Exam tip.** Concavity guarantees that *a* maximum found by gradient ascent is *the* maximum — not that it exists: if the two
    classes are perfectly separable the unregularised logistic weights grow without bound (Module 09 shows it; a
    Gaussian prior on the weights — MAP, an L2 penalty — fixes it). Know which course models are convex
    (least squares, ridge, lasso, logistic regression, SVM) and which are not (mixtures/EM, k-means, neural networks).

---
## 6 · Vector calculus: the derivative step of least-squares regression

> 📍 **Where we are.** Module 03 wrote least squares as $(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w)$ and
> solved the normal equations. The regression lecture gets there by differentiating with respect to the *whole
> vector* $\mathbf w$ at once. Here are the two rules it uses, why they are true, and a numerical check of every step.

🧮 **Math refresher — the gradient with respect to a vector.** For a scalar function $f(\mathbf w)$ of a vector
$\mathbf w = (w_1, \dots, w_J)^\top$, the derivative $\frac{\partial f}{\partial \mathbf w}$ is the gradient: the vector
of the $J$ partial derivatives. Two rules (eqs. 69 and 81 of *The Matrix Cookbook*, the standard reference for matrix
derivatives) cover everything in the regression lectures:

$$\frac{\partial\,(\mathbf a^\top\mathbf w)}{\partial\mathbf w} = \mathbf a, \qquad
\frac{\partial\,(\mathbf w^\top A\,\mathbf w)}{\partial\mathbf w} = (A + A^\top)\,\mathbf w \;\;(= 2A\mathbf w \text{ if } A \text{ is symmetric}).$$

*In words:* a linear function $a_1w_1 + \dots + a_Jw_J$ has slope $a_j$ in direction $j$ — just like $\frac{d}{dw}(aw) = a$. A quadratic form $\mathbf w^\top A\mathbf w = \sum_{j,k} A_{jk} w_j w_k$ behaves like $a w^2$, whose derivative is
$2aw$; each $w_j$ appears once in row $j$ and once in column $j$, which is why both $A$ and $A^\top$ show up.

**Check for $J = 2$.** $\mathbf w^\top A\mathbf w = A_{11}w_1^2 + (A_{12} + A_{21})w_1w_2 + A_{22}w_2^2$, so
$\partial/\partial w_1 = 2A_{11}w_1 + (A_{12}+A_{21})w_2$, which is the first entry of $(A + A^\top)\mathbf w$.

In [ ]:
# ── 6.1 Check both vector-calculus rules with finite differences ──
rng = np.random.default_rng(SEED + 6)
J = 4
a_vec = rng.normal(size=J)
A_mat = rng.normal(size=(J, J))                      # deliberately NOT symmetric
w_pt = rng.normal(size=J)
print("∂(aᵀw)/∂w   numerical:", numerical_gradient(lambda w: a_vec @ w, w_pt).round(6))
print("            rule a   :", a_vec.round(6))
print("∂(wᵀAw)/∂w  numerical:", numerical_gradient(lambda w: w @ A_mat @ w, w_pt).round(6))
print("            (A + Aᵀ)w:", ((A_mat + A_mat.T) @ w_pt).round(6))
print("            2Aw (wrong unless A is symmetric):", (2 * A_mat @ w_pt).round(6))

🔍 **What just happened?** Both rules match the finite differences, and the shortcut $2A\mathbf w$ is visibly wrong for a
non-symmetric $A$. In regression the matrix is $A = X^\top X$, which *is* symmetric, so $(A + A^\top)\mathbf w = 2X^\top X\mathbf w$ — the simplification used in the next derivation.

### 6.2 The normal equations, derived step by step

$$\begin{aligned}
\frac{\partial}{\partial\mathbf w}(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w)
&= \frac{\partial}{\partial\mathbf w}\big(\mathbf y^\top - \mathbf w^\top X^\top\big)\big(\mathbf y - X\mathbf w\big) && \text{transpose of a difference; } (X\mathbf w)^\top = \mathbf w^\top X^\top \\
&= \frac{\partial}{\partial\mathbf w}\big(\mathbf y^\top\mathbf y - \mathbf y^\top X\mathbf w - \mathbf w^\top X^\top\mathbf y + \mathbf w^\top X^\top X\mathbf w\big) && \text{distributive property} \\
&= \frac{\partial}{\partial\mathbf w}\big(-2\,\mathbf w^\top X^\top\mathbf y + \mathbf w^\top X^\top X\mathbf w\big) && \mathbf y^\top X\mathbf w = \mathbf w^\top X^\top\mathbf y \text{ (a number equals its transpose); } \mathbf y^\top\mathbf y \text{ has no } \mathbf w \\
&= -2X^\top\mathbf y + \big(X^\top X + (X^\top X)^\top\big)\mathbf w && \text{rule 1 with } \mathbf a = X^\top\mathbf y\text{; rule 2 with } A = X^\top X \\
&= -2X^\top\mathbf y + 2X^\top X\,\mathbf w = -2X^\top(\mathbf y - X\mathbf w) . && X^\top X \text{ is symmetric}
\end{aligned}$$

*In words:* expand the product, merge the two identical middle terms, drop the constant, and apply the two rules. The
gradient is $-2X^\top \times$ (residuals): it vanishes exactly when the residual vector is orthogonal to every column of
$X$ — Module 03's geometric picture. Setting it to zero gives the **normal equations**; multiplying both sides by
$(X^\top X)^{-1}$ gives the closed form:

$$X^\top X\,\mathbf w = X^\top\mathbf y \quad\Longrightarrow\quad \mathbf w = (X^\top X)^{-1}X^\top\mathbf y .$$

*In words:* the weights at which the slope is zero in every direction; because the objective is convex (its Hessian
$2X^\top X$ has non-negative eigenvalues), this flat point is the global minimum.

In [ ]:
# ── 6.2 Verify every claim on the prostate training men (8 standardised predictors + intercept) ──
train = prostate["train"].to_numpy()
Z_tr, _, _ = standardize(prostate.loc[train, FEATURES].to_numpy(float))
X_tr = np.column_stack([np.ones(train.sum()), Z_tr])                  # 67 × 9, augmented with a column of ones
y_tr = prostate.loc[train, "lpsa"].to_numpy(float)

def rss(w):
    """(y − Xw)ᵀ(y − Xw): the residual sum of squares."""
    r = y_tr - X_tr @ w
    return r @ r


w_rand = rng.normal(size=X_tr.shape[1])
print("max |formula − finite difference| of the gradient:",
      f"{np.abs(-2 * X_tr.T @ (y_tr - X_tr @ w_rand) - numerical_gradient(rss, w_rand)).max():.2e}")
w_ne = np.linalg.solve(X_tr.T @ X_tr, X_tr.T @ y_tr)                  # normal equations (Module 03: use solve)
print("gradient at the normal-equation solution:", (-2 * X_tr.T @ (y_tr - X_tr @ w_ne)).round(10) + 0.0)   # + 0.0: no "-0."
print("eigenvalues of the Hessian 2XᵀX:", np.linalg.eigvalsh(2 * X_tr.T @ X_tr).round(1), "→ all > 0: convex, unique minimum")

H_mean = 2 * X_tr.T @ X_tr / len(y_tr)                                # Hessian of the MEAN squared error
step = 1.0 / np.linalg.eigvalsh(H_mean).max()                        # a safe step size (below 2 / largest eigenvalue)
path = gradient_descent(lambda w: -2 * X_tr.T @ (y_tr - X_tr @ w) / len(y_tr), np.zeros(9), step=step,
                        n_steps=20000, f=lambda w: rss(w) / len(y_tr), tol=1e-14)
print(f"gradient descent (step {step:.3f}) stopped after {len(path) - 1} steps; max |w_GD − w_normal-eq| = "
      f"{np.abs(path[-1] - w_ne).max():.1e}")
print(pd.Series(w_ne, index=["w0"] + FEATURES).round(3).to_dict())

In [ ]:
# ── 6.3 How fast does gradient descent approach the closed form? ──
gap = np.array([rss(w) for w in path]) - rss(w_ne)
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.semilogy(np.maximum(gap, 1e-14), color="tab:blue")
ax.set_xlabel("gradient-descent iteration"); ax.set_ylabel("RSS(w_t) − RSS(w_closed form)")
ax.set_title("prostate training men: gradient descent converges to the normal-equation solution")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every step of the derivation checks out numerically: the gradient formula agrees with
finite differences to about $10^{-6}$ or better, it is zero at the normal-equation solution, and all Hessian eigenvalues
are positive (a strictly convex bowl, so the solution is unique). Gradient descent, started from $\mathbf w = \mathbf 0$
with a step size chosen from the largest Hessian eigenvalue, walks down to the same nine weights in 291 steps (largest
difference $6\times10^{-7}$); the gap in the residual sum of squares shrinks by a roughly constant factor per iteration
(close to a straight line on the log scale once the first fast steps are over) — the speed is set by the
ratio of the largest to the smallest Hessian eigenvalue (the condition number of Module 03). For least squares the
closed form is faster; gradient descent becomes the method of choice when no closed form exists (logistic regression,
neural networks) or when $J$ is so large that solving the $J\times J$ system is too costly (Module 03, Section 4.4).

### 6.4 The same derivation with a prior: ridge regression

The regression lecture's answer to a singular $X^\top X$ is MAP estimation with a Gaussian prior
$\mathbf w \sim \mathcal N(\mathbf 0, \sigma_0^2 I)$ on the regression coefficients (Module 05 explains MAP; the
lecture places no prior on the intercept $w_0$). The log-prior adds a penalty to the least-squares criterion:

$$\mathbf w_{\text{MAP}} = \arg\min_{\mathbf w}\Big[\frac{1}{2\sigma^2}(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w) + \frac{1}{2\sigma_0^2}\mathbf w^\top\mathbf w\Big]
= \arg\min_{\mathbf w}\Big[(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w) + \frac{\sigma^2}{\sigma_0^2}\mathbf w^\top\mathbf w\Big].$$

*In words:* multiply by $2\sigma^2$ (which does not move the minimum); the prior becomes a penalty on large weights with
strength $\lambda = \sigma^2/\sigma_0^2$. Differentiating with rule 2 once more ($A = \lambda I$, so the derivative of
$\lambda\mathbf w^\top\mathbf w$ is $2\lambda\mathbf w$) and inserting $\mathbf w = I\mathbf w$:

$$-2X^\top\mathbf y + 2X^\top X\mathbf w + \frac{2\sigma^2}{\sigma_0^2}\mathbf w = \mathbf 0
\;\Longrightarrow\; \Big(X^\top X + \frac{\sigma^2}{\sigma_0^2}I\Big)\mathbf w = X^\top\mathbf y
\;\Longrightarrow\; \mathbf w = \Big(X^\top X + \frac{\sigma^2}{\sigma_0^2}I\Big)^{-1}X^\top\mathbf y .$$

*In words:* the only change is a "ridge" of $\lambda = \sigma^2/\sigma_0^2$ added to the diagonal of $X^\top X$. A small
prior variance $\sigma_0^2$ (strong belief that the weights are near 0) means a large $\lambda$, and the matrix is
invertible even when $N < J$ (Module 03, Section 8).

✏️ **Your turn.** Add the ridge penalty $\lambda\,\mathbf w^\top\mathbf w$ (with, say, $\lambda = 5$) to `rss`, but do not
penalise $w_0$ (replace the first diagonal entry of $\lambda I$ by 0). Solve the modified normal equations, check that
the gradient of the penalised objective is zero there, and compare the eight weights with `w_ne`: which ones shrink
most? (Module 07 studies ridge regression in full.)

### 📘 Lecture notes — the normal equations (and ridge regression) by matrix calculus

**Two rules.** $\dfrac{\partial(\mathbf a^\top\mathbf w)}{\partial\mathbf w} = \mathbf a$ and
$\dfrac{\partial(\mathbf w^\top B\mathbf w)}{\partial\mathbf w} = (B + B^\top)\mathbf w$ ($= 2B\mathbf w$ for symmetric $B$).

**Least squares, line by line.** Starting from $\mathbf w_{\text{MCLE}} = \arg\min(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w)$:

1. move the transpose inside, flipping the order: $(\mathbf y^\top - \mathbf w^\top X^\top)(\mathbf y - X\mathbf w)$;
2. distributive property: $\mathbf y^\top\mathbf y - \mathbf y^\top X\mathbf w - \mathbf w^\top X^\top\mathbf y + \mathbf w^\top X^\top X\mathbf w$;
3. the two middle terms are the same scalar (a number equals its transpose) and $\mathbf y^\top\mathbf y$ has no
   $\mathbf w$: differentiate $-2\mathbf w^\top X^\top\mathbf y + \mathbf w^\top X^\top X\mathbf w$;
4. rules 1 and 2: $-2X^\top\mathbf y + \big(X^\top X + (X^\top X)^\top\big)\mathbf w = -2X^\top\mathbf y + 2X^\top X\mathbf w = \mathbf 0$;
5. normal equations $X^\top\mathbf y = X^\top X\mathbf w$; multiply by $(X^\top X)^{-1}$:
   $\mathbf w = (X^\top X)^{-1}X^\top\mathbf y$.

*In words:* the gradient is $-2X^\top$ times the residual vector; it vanishes when the residual is perpendicular to
every column of $X$. The Hessian $2X^\top X$ has no negative eigenvalues, so the flat point is the global minimum.

**Ridge regression = MAP with a Gaussian prior.** With $\mathbf w \sim \mathcal N(\mathbf 0, \sigma_0^2 I)$ (no prior on
$w_0$): minimise $(\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w) + \frac{\sigma^2}{\sigma_0^2}\mathbf w^\top\mathbf w$;
the same five steps give

$$\mathbf w_{\text{MAP}} = \Big(X^\top X + \frac{\sigma^2}{\sigma_0^2}I\Big)^{-1}X^\top\mathbf y,$$

*in words:* a small $\sigma_0^2$ (strong prior belief that weights are small) adds a large ridge to the diagonal;
the matrix is invertible even if $N < J$.

**Worked check (Section 6.2).** On the 67 prostate training men ($J + 1 = 9$ weights) the analytic gradient agrees
with finite differences, is zero at the normal-equation solution, and gradient descent converges to the same weights.

> 🎓 **Exam tip.** This derivation is a classic exam question: be able to reproduce it from memory, including the reason the two
    cross terms combine ("both are the same scalar") and the factor 2 from $(A + A^\top)$ with $A$ symmetric. The ridge
    version is the same derivation with an extra $\lambda\mathbf w^\top\mathbf w$, $\lambda = \sigma^2/\sigma_0^2$.

---
## 7 · Constrained optimisation and Lagrange multipliers

> 📍 **Where we are.** So far the parameters could take any value. Often they cannot: cell-type fractions must add up
> to one, a direction in PCA must have length one, and an SVM's boundary must keep every training sample on the
> correct side of its margin. **Lagrange multipliers** turn such constrained problems into unconstrained ones.

### 7.1 Equality constraints: why PCA ends in an eigenvector

PCA as an optimisation problem: the projection of a sample $\mathbf x$ onto an axis $\mathbf u$ is $\mathbf x^\top\mathbf u$.
For a mean-centred $N\times K$ data matrix $X$, find the unit vector $\mathbf u$ along which the projected values have
the largest variance (Module 03, Section 6.3: that variance is $\mathbf u^\top S\mathbf u$),

$$\max_{\mathbf u}\; \mathbf u^\top S\,\mathbf u = \frac1N\,\mathbf u^\top X^\top X\,\mathbf u \qquad \text{subject to} \qquad \mathbf u^\top\mathbf u = 1 .$$

*In words:* without the constraint we could make the variance as large as we like simply by making $\mathbf u$ longer;
the constraint says only the *direction* counts.

🧮 **Math refresher — the Lagrangian.** To maximise $f(\mathbf u)$ subject to $g(\mathbf u) = c$, introduce a new
unknown $\lambda$ (the **Lagrange multiplier**) and look for flat points of

$$\mathcal L(\mathbf u, \lambda) = f(\mathbf u) + \lambda\,\big(c - g(\mathbf u)\big), \qquad \nabla_{\mathbf u}\mathcal L = \nabla f - \lambda\nabla g = \mathbf 0 .$$

*In words:* at the constrained optimum you cannot improve $f$ by sliding along the constraint surface, which means the
uphill direction of $f$ must be perpendicular to that surface — parallel to the gradient of $g$. The multiplier
$\lambda$ is the proportionality factor between the two gradients. (Setting $\partial\mathcal L/\partial\lambda = 0$
gives back the constraint.)

For PCA the Lagrangian is $\mathcal L = \frac1N\mathbf u^\top X^\top X\mathbf u + \lambda(1 - \mathbf u^\top\mathbf u)$.
With rule 2 of Section 6 ($A = S$ symmetric, and $A = I$):

$$\nabla_{\mathbf u}\mathcal L = 2S\mathbf u - 2\lambda\mathbf u = \mathbf 0 \;\Longleftrightarrow\; S\mathbf u - \lambda\mathbf u = \mathbf 0 \;\Longleftrightarrow\; S\mathbf u = \lambda\mathbf u .$$

*In words:* as $\mathbf u \ne \mathbf 0$, the best direction must be an **eigenvector** of the covariance matrix
$S = \frac1N X^\top X$, with the multiplier as its eigenvalue. (The PCA lecture writes
$\frac1N X^\top X\mathbf u - \lambda\mathbf u = 0$, i.e. it has already divided by 2.) Multiplying $S\mathbf u = \lambda\mathbf u$ by
$\mathbf u^\top$ gives $\mathbf u^\top S\mathbf u = \lambda$: the multiplier *is* the variance, so the maximum is the
eigenvector with the **largest** eigenvalue.

**Proof that no other unit vector does better.** Any unit vector can be written in the eigenvector basis,
$\mathbf u = \sum_{k=1}^{K} a_k\mathbf u_k$ with $\mathbf u^\top\mathbf u = \sum_k a_k^2 = 1$ (Module 03, Section 7). Then

$$\mathbf u^\top S\mathbf u = \Big(\sum_k a_k\mathbf u_k\Big)^\top S\Big(\sum_k a_k\mathbf u_k\Big) = \sum_{k=1}^{K} a_k^2\lambda_k \;\le\; \lambda_1\sum_k a_k^2 = \lambda_1 = \mathbf u_1^\top S\mathbf u_1 ,$$

*in words:* the variance along $\mathbf u$ is a weighted average of the eigenvalues with weights $a_k^2$ that add up to
1, which can never exceed the largest eigenvalue $\lambda_1$; it reaches $\lambda_1$ only along $\mathbf u_1$. So
$\mathbf u_1$ is the first PC direction. **The other PCs follow the same logic:** given $\mathbf u_1$, the largest
variance among all directions orthogonal to $\mathbf u_1$ is $\lambda_2 = \mathbf u_2^\top S\mathbf u_2$, so $\mathbf u_2$
is PC 2; given $\mathbf u_1, \mathbf u_2$, the best orthogonal direction is $\mathbf u_3$, and so on.

In [ ]:
# ── 7.1 Brute force over all directions vs the eigenvector (SIMULATED leukaemia data: TCF3 and CCND3) ──
X2 = leuk["X"][:, [GENE_INDEX["TCF3"], GENE_INDEX["CCND3"]]]
X2c = X2 - X2.mean(axis=0)
S2 = X2c.T @ X2c / len(X2c)
thetas = np.linspace(0, np.pi, 1801)                       # every direction u(θ) = (cos θ, sin θ) has length 1
U_theta = np.c_[np.cos(thetas), np.sin(thetas)]
var_theta = np.einsum("ij,jk,ik->i", U_theta, S2, U_theta) # uᵀ S u for every θ
lam, U = np.linalg.eigh(S2)
u1 = U[:, -1] * np.sign(U[0, -1])                          # largest eigenvalue's eigenvector (eigh is ascending)
theta_star = thetas[var_theta.argmax()]
print(f"brute force: best θ = {np.degrees(theta_star):.2f}°, max variance = {var_theta.max():.5f}, min = {var_theta.min():.5f}")
print(f"eigh       : u1 at θ = {np.degrees(np.arctan2(u1[1], u1[0])):.2f}°, eigenvalues λ1 = {lam[-1]:.5f}, λ2 = {lam[0]:.5f}")
grad_f, grad_g = 2 * S2 @ u1, 2 * u1                       # gradients of the objective and of the constraint at u1
print("at u1: ∇f =", grad_f.round(5), " ∇g =", grad_g.round(5), " ratio ∇f/∇g =", (grad_f / grad_g).round(5), "= λ1")

# the same answer from a general-purpose constrained optimiser, for 5 genes at once
idx5 = np.argsort(leuk["X"].var(axis=0))[::-1][:5]
X5c = leuk["X"][:, idx5] - leuk["X"][:, idx5].mean(axis=0); S5 = X5c.T @ X5c / len(X5c)
res = minimize(lambda u: -u @ S5 @ u, x0=np.ones(5) / np.sqrt(5), method="SLSQP",
               constraints=[{"type": "eq", "fun": lambda u: u @ u - 1}])
top5 = np.linalg.eigh(S5)[1][:, -1]
print(f"5 genes: SLSQP variance {-res.fun:.5f} vs largest eigenvalue {np.linalg.eigvalsh(S5)[-1]:.5f}; "
      f"|cos(angle)| between solutions = {abs(res.x @ top5) / np.linalg.norm(res.x):.6f}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(np.degrees(thetas), var_theta, color="tab:blue")
for v, lab, c in [(lam[-1], "λ1 (largest eigenvalue)", "tab:red"), (lam[0], "λ2", "tab:green")]:
    ax.axhline(v, ls="--", lw=1, color=c, label=lab)
ax.axvline(np.degrees(theta_star), color="0.5", ls=":", lw=1)
ax.set_xlabel("direction θ of the unit vector u (degrees)"); ax.set_ylabel("variance uᵀSu")
ax.set_title("SIMULATED leukaemia, TCF3 vs CCND3: the best direction is the top eigenvector"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Trying all 1,801 directions, the largest projected variance occurs at the angle of the top
eigenvector and equals $\lambda_1$; the smallest equals $\lambda_2$. At $\mathbf u_1$ the gradient of the objective is
exactly $\lambda_1$ times the gradient of the constraint — the Lagrange condition, with the multiplier equal to the
variance. For five genes, where brute force over directions is no longer practical, SciPy's general constrained
optimiser (SLSQP) finds the same direction (|cosine| ≈ 1) and the same maximum as the eigendecomposition. Module 14 will
use this result to compute all principal components at once.

### 7.2 Inequality constraints: the primal–dual toy problem

The SVM lecture solves its optimisation problem through a **dual formulation**, because the dual identifies the support
vectors and makes the kernel trick possible (Module 11). It introduces duality with the smallest possible example —
"in general, Lagrange multipliers can be applied to turn the original primal problem into a dual problem":

$$\text{primal: } \min_x\; x^2 \;\;\text{s.t. } x \ge b; \qquad
\text{Lagrangian: } \min_x \max_{\alpha \ge 0}\; x^2 - \alpha(x - b); \qquad
\text{dual: } \max_{\alpha\ge 0}\; -\tfrac{\alpha^2}{4} + \alpha b .$$

*In words:* we want the smallest $x^2$ while keeping $x$ at least $b$. Each constraint gets a multiplier $\alpha \ge 0$
(one per training sample in the full SVM of Section 7.3). The Lagrangian reproduces the constraint automatically: for fixed $x$,
if $x < b$ the term $-\alpha(x-b)$ is positive and the inner maximum over $\alpha$ is $+\infty$ (a huge penalty); if
$x \ge b$ the best $\alpha$ is 0 and we are left with $x^2$. So the min–max equals the primal.

**Deriving the dual.** Swap the order: for fixed $\alpha$, minimise over $x$:
$\frac{d}{dx}[x^2 - \alpha(x-b)] = 2x - \alpha = 0$ gives $x = \alpha/2$, and substituting,

$$g(\alpha) = \Big(\frac{\alpha}{2}\Big)^2 - \alpha\Big(\frac{\alpha}{2} - b\Big) = -\frac{\alpha^2}{4} + \alpha b .$$

*In words:* the **dual function** $g(\alpha)$ is the best the primal variable can do for a given price $\alpha$ on the
constraint. Maximising it over $\alpha \ge 0$: $g'(\alpha) = -\alpha/2 + b = 0$ gives $\alpha^\ast = 2b$ when $b > 0$
(constraint **active**: $x^\ast = b$), and $\alpha^\ast = 0$ when $b \le 0$ (constraint **inactive**: $x^\ast = 0$).
In both cases $\alpha^\ast (x^\ast - b) = 0$ (**complementary slackness**): a multiplier can be non-zero only if its
constraint is tight. In the SVM, the samples with $\alpha_i > 0$ — tight constraints — are the **support vectors**.

In [ ]:
# ── 7.2 Solve the primal–dual toy by hand, by grid search and with SciPy, for b = 1 and b = −1 ──
def solve_toy(b):
    """Return (x*, α*, primal value, dual value) computed three ways for: min x² s.t. x ≥ b."""
    x_hand, a_hand = max(b, 0.0), max(2 * b, 0.0)                       # from the derivation above
    xs = np.linspace(-3, 3, 60001); feas = xs >= b
    x_grid = xs[feas][np.argmin(xs[feas] ** 2)]                          # primal by grid search
    al = np.linspace(0, 6, 60001); g = -al ** 2 / 4 + al * b
    a_grid = al[np.argmax(g)]                                            # dual by grid search
    sp = minimize(lambda x: x[0] ** 2, x0=[2.5], method="SLSQP", constraints=[{"type": "ineq", "fun": lambda x: x[0] - b}])
    print(f"b = {b:+}: x* hand {x_hand:+.3f} | grid {x_grid:+.3f} | SciPy {round(sp.x[0], 3) + 0.0:+.3f};   α* hand {a_hand:.3f} | grid {a_grid:.3f};"
          f"   primal value {x_hand ** 2:.3f} = dual value {round(-a_hand ** 2 / 4 + a_hand * b, 3) + 0.0:.3f};"
          f"   α*(x* − b) = {a_hand * (x_hand - b):.1f}")
    return xs, al, g


xs1, al1, g1 = solve_toy(1.0)      # constraint active
solve_toy(-1.0)                    # constraint inactive
print("weak duality: g(α) ≤ primal optimum for every α ≥ 0?", bool(np.all(g1 <= 1.0 + 1e-12)))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].plot(xs1, xs1 ** 2, color="tab:blue"); axes[0].axvspan(-3, 1, color="tab:red", alpha=0.12, label="infeasible: x < b = 1")
axes[0].plot(1, 1, "k*", ms=12, label="x* = b = 1 (constraint active)"); axes[0].set_xlim(-2.5, 2.5); axes[0].set_ylim(0, 6)
axes[0].set_xlabel("primal variable x"); axes[0].set_ylabel("x²"); axes[0].set_title("primal: min x² s.t. x ≥ 1"); axes[0].legend(fontsize=7)
axes[1].plot(al1, g1, color="tab:green"); axes[1].plot(2, 1, "k*", ms=12, label="α* = 2b = 2, g(α*) = 1")
axes[1].axhline(1, color="0.5", ls="--", lw=1, label="primal optimum = 1")
axes[1].set_xlabel("dual variable α (≥ 0)"); axes[1].set_ylabel("g(α) = −α²/4 + αb"); axes[1].set_ylim(-3, 1.6)
axes[1].set_title("dual: max g(α) s.t. α ≥ 0"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For $b = 1$ all three methods agree: $x^\ast = 1$ (the constraint pushes $x$ away from the
unconstrained minimum 0), $\alpha^\ast = 2$, and the primal and dual optimal values are both 1 — **strong duality**. For
$b = -1$ the constraint is harmless, $x^\ast = 0$ and $\alpha^\ast = 0$. In both cases $\alpha^\ast(x^\ast - b) = 0$.
The right panel also shows **weak duality**: every value of the dual function lies at or below the primal optimum, so
any feasible $\alpha$ gives a lower bound and the dual's maximum touches the primal's minimum.

⚠️ **Two common confusions.** (1) *Why may we swap "min" and "max"?* The precise statement is:
$\max_{\alpha\ge0}\min_x \mathcal L \le \min_x\max_{\alpha\ge0}\mathcal L$ always holds (weak duality), and for convex
problems such as this one and the SVM the two sides are equal (strong duality) — which is what the computation above
confirms. (2) *Which sign do the multipliers have?* With the Lagrangian written as "objective $-\ \sum_i \alpha_i \times$
(constraint written as $\ge 0$)", as here and in the SVM ($\frac12\mathbf w^\top\mathbf w - \sum_i\alpha_i[\dots]$), the
multipliers must satisfy $\alpha_i \ge 0$; you will occasionally see "$\alpha_i \le 0$" written by mistake.

✏️ **Your turn.** Change the primal to $\min_x (x - 3)^2$ s.t. $x \le 1$ (rewrite the constraint as $1 - x \ge 0$).
Write the Lagrangian, derive $g(\alpha)$, find $\alpha^\ast$ and $x^\ast$ by hand, and check with `minimize`.

### 📘 Lecture notes — PCA as a constrained optimisation

**The problem.** The projection of a sample $\mathbf x$ onto an axis $\mathbf u$ is $\mathbf x^\top\mathbf u$. For a
mean-centred $N\times K$ data matrix $X$, find a unit vector $\mathbf u$ such that the variance of the projected values
is greatest:

$$\text{maximise } \frac1N\mathbf u^\top X^\top X\mathbf u \quad\text{subject to}\quad \mathbf u^\top\mathbf u = 1 .$$

*In words:* the constraint makes sure $\mathbf u$ is a unit vector — without it the objective grows without bound
as $\mathbf u$ gets longer.

**How to solve it.** Construct the Lagrangian $\frac1N\mathbf u^\top X^\top X\mathbf u + \lambda(1 - \mathbf u^\top\mathbf u)$,
differentiate with respect to $\mathbf u$ and set to zero:

$$\frac1N X^\top X\mathbf u - \lambda\mathbf u = \mathbf 0 \quad\Longleftrightarrow\quad S\mathbf u - \lambda\mathbf u = \mathbf 0,
\qquad S = \frac1N X^\top X \text{ (the covariance matrix)} .$$

*In words:* as $\mathbf u \ne \mathbf 0$, $\mathbf u$ must be an eigenvector of $S$ with eigenvalue $\lambda$ (the
factor 2 from differentiating has been divided out).

**Which eigenvector?** At an eigenvector the objective equals $\mathbf u_k^\top S\mathbf u_k = \lambda_k$, so take the
largest: $\mathbf u_1^\top S\mathbf u_1 = \lambda_1$ is the largest possible value — writing any unit vector as
$\mathbf u = \sum_k a_k\mathbf u_k$ ($\sum_k a_k^2 = 1$) gives $\mathbf u^\top S\mathbf u = \sum_k a_k^2\lambda_k \le \lambda_1$.
Thus $\mathbf u_1$ is the first PC direction; given $\mathbf u_1$, the best direction orthogonal to it is $\mathbf u_2$
(variance $\lambda_2$), the second PC; given $\mathbf u_1, \mathbf u_2$, the third is $\mathbf u_3$; and so on.

**The Lagrange condition in pictures.** At the constrained optimum the gradient of the objective ($2S\mathbf u$) is
parallel to the gradient of the constraint ($2\mathbf u$), and the ratio is the multiplier $\lambda$ — checked
numerically in Section 7.1.

> 🎓 **Exam tip.** Be able to write the Lagrangian, differentiate it with $\partial(\mathbf u^\top S\mathbf u)/\partial\mathbf u = 2S\mathbf u$,
    and explain why the answer is the eigenvector with the *largest* eigenvalue (the objective value equals the
    multiplier). Without the constraint the problem has no solution (unbounded).

### 7.3 The support vector machine: primal, dual and support vectors

The SVM (Module 11) looks for the linear boundary $\mathbf w^\top\mathbf x + b = 0$ between two classes $y_i\in\{-1, +1\}$
that leaves the **largest margin** to the nearest points of both classes. Instead of fitting all points, it focuses on
the boundary points — the **support vectors** that "support" the decision boundary. The margin width is
$M = 2/\|\mathbf w\|$, and maximising $M$ means minimising its inverse, which gives the **primal** quadratic programme

$$\min_{\mathbf w, b}\ \frac12\mathbf w^\top\mathbf w \qquad\text{subject to}\qquad y_i(\mathbf w^\top\mathbf x_i + b) \ge 1 \quad (i = 1, \dots, n) .$$

*In words:* the smallest weight vector that puts every training sample on the correct side, at least one margin unit
away from the boundary — $n$ constraints for $n$ training samples, each with its own multiplier $\alpha_i \ge 0$.

**The Lagrangian and the dual.** Exactly as in the toy problem:

$$\mathcal L(\mathbf w, b, \boldsymbol\alpha) = \frac12\mathbf w^\top\mathbf w - \sum_{i=1}^{n}\alpha_i\big[y_i(\mathbf w^\top\mathbf x_i + b) - 1\big],
\qquad \frac{\partial\mathcal L}{\partial\mathbf w} = \mathbf 0 \Rightarrow \mathbf w = \sum_i\alpha_i y_i\mathbf x_i,
\qquad \frac{\partial\mathcal L}{\partial b} = 0 \Rightarrow \sum_i \alpha_i y_i = 0 .$$

*In words:* the optimal weight vector is a weighted sum of training samples; substituting it back into $\mathcal L$
leaves a problem in the multipliers alone, the **dual**:

$$\max_{\boldsymbol\alpha}\ \sum_i\alpha_i - \frac12\sum_{i}\sum_{j}\alpha_i\alpha_j y_iy_j\,\mathbf x_i^\top\mathbf x_j
\qquad\text{subject to}\qquad \alpha_i \ge 0,\quad \sum_i\alpha_iy_i = 0 .$$

*In words:* the data enter only through dot products $\mathbf x_i^\top\mathbf x_j$ between training samples (the door
to the kernel trick). By complementary slackness, $\alpha_i > 0$ only for samples whose constraint is tight — the
points exactly on the margin, the **support vectors**; $\mathbf w$ is defined by them alone. A new sample $\mathbf x$ is
classified by the sign of $\mathbf w^\top\mathbf x + b = \sum_{i\in SV}\alpha_iy_i\,\mathbf x_i^\top\mathbf x + b$: dot
products with the support vectors only.

**Non-separable data (soft margin).** If no line separates the classes, each sample gets a slack variable
$\varepsilon_i \ge 0$ and the primal becomes $\min \frac12\mathbf w^\top\mathbf w + C\sum_i\varepsilon_i$ s.t.
$y_i(\mathbf w^\top\mathbf x_i + b) \ge 1 - \varepsilon_i$. The dual is the same except that the multipliers are now
bounded, $0 \le \alpha_i \le C$: the constant $C$ stops any single $\alpha_i$ from going to infinity, which prevents the
model from overfitting to noisy samples. Support vectors are then the points on the margin ($0 < \alpha_i < C$) and the
points on the wrong side of their margin ($\alpha_i = C$).

We solve both formulations for twelve SIMULATED tumours described by two standardised marker genes, and then add one
mislabelled tumour.

In [ ]:
# ── 7.3 Hard-margin SVM: primal and dual solved with SciPy; then a soft margin (SIMULATED tumours) ──
rng = np.random.default_rng(SEED + 73)
X_sv = np.vstack([rng.normal([-1.2, -1.0], 0.55, size=(6, 2)), rng.normal([1.2, 1.0], 0.55, size=(6, 2))])
y_sv = np.r_[-np.ones(6), np.ones(6)]                     # −1 = tumour type A, +1 = tumour type B

def svm_primal(X, y):
    """min ½ wᵀw  s.t.  y_i (wᵀx_i + b) ≥ 1, solved directly for (w, b)."""
    cons = [{"type": "ineq", "fun": lambda p, i=i: y[i] * (X[i] @ p[:2] + p[2]) - 1} for i in range(len(y))]
    res = minimize(lambda p: 0.5 * p[:2] @ p[:2], x0=np.zeros(3), method="SLSQP", constraints=cons)
    return res.x[:2], res.x[2]

def svm_dual(X, y, C=np.inf):
    """max Σα − ½ ΣΣ α_i α_j y_i y_j x_iᵀx_j  s.t.  0 ≤ α_i ≤ C, Σ α_i y_i = 0; returns (α, w, b)."""
    Q = (y[:, None] * X) @ (y[:, None] * X).T              # Q_ij = y_i y_j x_iᵀx_j
    res = minimize(lambda a: 0.5 * a @ Q @ a - a.sum(), x0=np.zeros(len(y)), jac=lambda a: Q @ a - 1,
                   method="SLSQP", bounds=[(0, None if np.isinf(C) else C)] * len(y),
                   constraints=[{"type": "eq", "fun": lambda a: a @ y, "jac": lambda a: y}],
                   options={"ftol": 1e-12, "maxiter": 500})
    a = np.where(res.x > 1e-6, res.x, 0.0)                 # numerical zeros → exact zeros
    w = (a * y) @ X                                        # w = Σ α_i y_i x_i
    on_margin = (a > 1e-6) & (a < C - 1e-6)                # 0 < α_i < C: exactly on the margin
    b = np.mean(y[on_margin] - X[on_margin] @ w)           # there y_i (wᵀx_i + b) = 1
    return a, w, b


w_p, b_p = svm_primal(X_sv, y_sv)
a_d, w_d, b_d = svm_dual(X_sv, y_sv)
sv = np.flatnonzero(a_d > 0)
print(f"primal: w = {w_p.round(4)}, b = {b_p:.4f}, margin 2/‖w‖ = {2 / np.linalg.norm(w_p):.3f}")
print(f"dual  : w = Σ α_i y_i x_i = {w_d.round(4)}, b = {b_d:.4f}")
print("α =", a_d.round(3), "| Σ α_i y_i =", round(float(a_d @ y_sv), 10) + 0.0)
print("support vectors (α > 0): samples", sv.tolist(), "| their y_i(wᵀx_i + b) =", (y_sv[sv] * (X_sv[sv] @ w_d + b_d)).round(3))
print("smallest y_i(wᵀx_i + b) among the others:", round(float(np.min(np.delete(y_sv * (X_sv @ w_d + b_d), sv))), 3))
x_new = np.array([0.3, -0.2])
print(f"new tumour x = {x_new}: primal wᵀx + b = {x_new @ w_p + b_p:+.4f} | dual Σ_SV α_i y_i x_iᵀx + b = "
      f"{(a_d[sv] * y_sv[sv]) @ (X_sv[sv] @ x_new) + b_d:+.4f}  ({len(sv)} dot products instead of {len(y_sv)})")

X_soft = np.vstack([X_sv, [[0.9, 0.6]]]); y_soft = np.r_[y_sv, -1.0]   # one type-A tumour deep among type B
a_s, w_s, b_s = svm_dual(X_soft, y_soft, C=1.0)
print(f"\nsoft margin, C = 1: α = {a_s.round(3)}")
print("   on the margin (0 < α < C):", np.flatnonzero((a_s > 0) & (a_s < 1 - 1e-6)).tolist(),
      "| at the bound (α = C):", np.flatnonzero(a_s >= 1 - 1e-6).tolist(),
      f"| y(wᵀx + b) of the mislabelled tumour = {y_soft[-1] * (X_soft[-1] @ w_s + b_s):+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
gx = np.linspace(-3, 3, 200)
for ax, (X_, y_, a_, w_, b_, title) in zip(axes, [(X_sv, y_sv, a_d, w_d, b_d, "hard margin: 3 support vectors"),
                                                    (X_soft, y_soft, a_s, w_s, b_s, "soft margin (C = 1): a mislabelled tumour")]):
    for lab, col in [(-1, "tab:blue"), (1, "tab:red")]:
        ax.scatter(*X_[y_ == lab].T, color=col, s=30, label=f"type {'A' if lab < 0 else 'B'} (y = {lab:+d})", zorder=3)
    s_ = a_ > 0
    ax.scatter(*X_[s_].T, s=170, facecolors="none", edgecolors="k", lw=1.3, label="support vector (α > 0)", zorder=4)
    for off, ls in [(0, "-"), (1, "--"), (-1, "--")]:                # wᵀx + b = 0, ±1
        ax.plot(gx, (off - b_ - w_[0] * gx) / w_[1], "k", ls=ls, lw=1.2 if off == 0 else 0.8)
    for i in np.flatnonzero(s_):
        ax.annotate(f"α={a_[i]:.2f}", X_[i], xytext=(-22, 10) if X_[i, 0] > 1.5 else (8, -14), textcoords="offset points",
                    fontsize=7)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    ax.set_xlabel("marker gene 1 (standardised)"); ax.set_ylabel("marker gene 2 (standardised)")
    ax.set_title(f"SIMULATED tumours, {title}", fontsize=9)
axes[0].legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The primal (solved for $\mathbf w, b$ directly) and the dual (solved for the twelve
multipliers) give the same boundary: $\mathbf w \approx (0.507, 0.534)$, $b \approx 0.162$, margin
$2/\|\mathbf w\| \approx 2.72$ standardised units. Only **three** of the twelve multipliers are non-zero (samples 1, 4 and
9 — two of type A and one of type B), they satisfy $\sum_i\alpha_iy_i = 0$, and exactly these three points lie on the
margin, $y_i(\mathbf w^\top\mathbf x_i + b) = 1$; every other tumour is further away (at least 1.06) and could be deleted
without changing the classifier. For a new tumour the primal score $\mathbf w^\top\mathbf x + b$ and the dual score built
from three dot products with the support vectors agree. This is the parameter trade-off of the lecture: the primal has
one parameter per **feature** ($\mathbf w, b$: the decision boundary and margin), the dual one per **training sample**
($\alpha_i$: which samples are support vectors), related by $\mathbf w = \sum_i\alpha_iy_i\mathbf x_i$. Predicting with
primal variables costs $m$ operations for $m$ features; with dual variables $m\,r$ operations for $r$ support vectors.

With a mislabelled tumour added, no line separates the classes; the soft-margin dual with $C = 1$ still has a solution.
The mislabelled tumour sits on the wrong side ($y(\mathbf w^\top\mathbf x + b) \approx -0.62$) with its multiplier at the
bound $\alpha = C = 1$, three tumours lie exactly on the new margin with $0 < \alpha < C$, and all others have
$\alpha = 0$. The bound $C$ caps the influence any single noisy sample can have on the boundary.

✏️ **Your turn.** Rerun the soft-margin fit with $C = 0.1$ and $C = 100$. How do the margin width $2/\|\mathbf w\|$ and
the number of support vectors change? Which value lets the mislabelled tumour pull the boundary most?

### 📘 Lecture notes — Lagrangian duality and support vectors (the SVM lecture)

**The toy problem.** Primal $\min_x x^2$ s.t. $x \ge b$ (primal parameter $x$). Lagrangian
$\min_x\max_{\alpha\ge0}\ x^2 - \alpha(x - b)$ (dual parameter $\alpha$, one per constraint). Setting
$\frac{d}{dx}[x^2 - \alpha(x - b)] = 0$ gives $x = \alpha/2$; substituting gives the dual
$\max_{\alpha\ge0} -\alpha^2/4 + \alpha b$. Solution: $\alpha^\ast = 2b$, $x^\ast = b$ if $b > 0$ (active constraint);
$\alpha^\ast = 0$, $x^\ast = 0$ if $b \le 0$ (inactive). Always $\alpha^\ast(x^\ast - b) = 0$ (complementary slackness)
and primal optimum = dual optimum (strong duality, for convex problems).

**The recipe.** Lagrangian → minimise over the primal variables (derivative = 0) → substitute → maximise the dual over
$\alpha \ge 0$.

**The SVM.** Maximising the margin $M = 2/\|\mathbf w\|$ means

$$\min_{\mathbf w,b}\ \tfrac12\mathbf w^\top\mathbf w \ \ \text{s.t.}\ \ y_i(\mathbf w^\top\mathbf x_i + b) \ge 1\ \ (n \text{ constraints}),
\qquad \max_{\boldsymbol\alpha}\ \sum_i\alpha_i - \tfrac12\sum_{i,j}\alpha_i\alpha_jy_iy_j\mathbf x_i^\top\mathbf x_j\ \ \text{s.t.}\ \ \alpha_i \ge 0,\ \sum_i\alpha_iy_i = 0 .$$

*In words:* differentiating the Lagrangian with respect to $\mathbf w$ and $b$ gives $\mathbf w = \sum_i\alpha_iy_i\mathbf x_i$
and $\sum_i\alpha_iy_i = 0$; $\mathbf w$ is defined only by the samples with $\alpha_i \ne 0$, the **support vectors**
(each data point has its own $\alpha_i$).

| | primal | dual |
|---|---|---|
| parameters | $\mathbf w, b$ | $\alpha_1, \dots, \alpha_n$ |
| number | number of features (+1) | number of training samples $n$ |
| meaning | decision boundary and margin | which samples are support vectors |
| cost of classifying a new sample | $m$ operations ($m$ features) | $m\,r$ operations ($r$ support vectors) |

**Soft margin.** Same dual, but $0 \le \alpha_i \le C$; $C$ keeps any $\alpha_i$ from going to infinity and so prevents
overfitting to noisy samples. Support vectors: points on the margin ($0 < \alpha_i < C$) and points on the wrong side
of their margin ($\alpha_i = C$).

**Worked example (Section 7.3).** Twelve tumours, two genes: 3 support vectors define $\mathbf w$; adding a
mislabelled tumour gives it $\alpha = C$.

> 🎓 **Exam tip.** Know the recipe (Lagrangian → $\partial/\partial$ primal = 0 → substitute → maximise over $\alpha \ge 0$), the
    result $\mathbf w = \sum_i \alpha_i y_i \mathbf x_i$, and why complementary slackness makes most $\alpha_i$ zero. A
    frequent question: "how many parameters do the primal and the dual have, and when is the dual cheaper?" (dual: one per
    training sample; cheaper to predict with when there are few support vectors and especially with kernels).

---
## 8 · Coordinate descent: one parameter at a time

> 📍 **Where we are.** Gradient descent moves all parameters at once. A different strategy is to optimise **one
> coordinate at a time**, holding all others fixed. The 02-620 lasso is fitted this way — by the **pathwise coordinate
> descent** algorithm of Friedman *et al.* (2007): in each iteration update each coefficient $\beta_j$ in turn, and
> iterate until convergence (fast implementations exist in the R package `glmnet` and in scikit-learn; Module 07). The
> same "one at a time" logic reappears in EM (Module 13) and Gibbs sampling (Module 21).

For least squares, fixing every weight except $w_j$ leaves a one-dimensional parabola in $w_j$. Setting its derivative
to zero (Section 2's recipe) gives a closed-form update:

$$w_j \leftarrow \frac{\mathbf x_j^\top\,\mathbf r_{(-j)}}{\mathbf x_j^\top\mathbf x_j}, \qquad \mathbf r_{(-j)} = \mathbf y - \sum_{k\ne j}\mathbf x_k w_k .$$

*In words:* compute the residual the *other* features leave unexplained, and regress it on feature $j$ alone. Cycling
through $j = 1, \dots, J$ repeatedly ("sweeps") can only lower the error at every step and, for a convex problem like
this one, converges to the same solution as the normal equations. The lasso changes a single line: the update is passed
through a "soft-threshold" that sets small weights exactly to zero (Module 07).

In [ ]:
# ── 8.1 Coordinate descent on the prostate data ──
def coordinate_descent(X, y, n_sweeps, w_start=None, record=False):
    """Cyclic coordinate descent for min ‖y − Xw‖² (no intercept: centre X and y first)."""
    w = np.zeros(X.shape[1]) if w_start is None else np.array(w_start, float)
    path = [w.copy()]
    for _ in range(n_sweeps):
        for j in range(X.shape[1]):
            r_minus_j = y - X @ w + X[:, j] * w[j]          # residual without feature j's contribution
            w[j] = X[:, j] @ r_minus_j / (X[:, j] @ X[:, j])
            if record:
                path.append(w.copy())
    return (w, np.array(path)) if record else w


Zc = Z_tr - Z_tr.mean(axis=0); yc = y_tr - y_tr.mean()          # centred: the intercept is just mean(y)
w_ols = np.linalg.solve(Zc.T @ Zc, Zc.T @ yc)
for sweeps in [1, 5, 20, 100, 500]:
    w_cd = coordinate_descent(Zc, yc, sweeps)
    print(f"{sweeps:3d} sweeps: max |w_CD − w_OLS| = {np.abs(w_cd - w_ols).max():.2e}")
print("OLS weights (same as Section 6 without w0):", w_ols.round(3))

two = [FEATURES.index("lcavol"), FEATURES.index("lcp")]          # two correlated predictors (r ≈ 0.59)
X_two = Zc[:, two]
w_two, path_two = coordinate_descent(X_two, yc, 8, w_start=[-0.6, 1.2], record=True)
A1, A2 = np.meshgrid(np.linspace(-0.8, 1.3, 150), np.linspace(-0.8, 1.4, 150))
R = np.mean((yc[None, None, :] - A1[..., None] * X_two[:, 0] - A2[..., None] * X_two[:, 1]) ** 2, axis=-1)
fig, ax = plt.subplots(figsize=(5.2, 4))
ax.contour(A1, A2, R, levels=np.geomspace(R.min() * 1.01, R.max(), 15), cmap="viridis", linewidths=0.8)
ax.plot(path_two[:, 0], path_two[:, 1], "o-", color="tab:red", ms=3, lw=1.2, label="coordinate descent")
ax.plot(*np.linalg.solve(X_two.T @ X_two, X_two.T @ yc), "k*", ms=12, label="least-squares solution")
ax.set_xlabel("weight of lcavol (standardised)"); ax.set_ylabel("weight of lcp (standardised)")
ax.set_title("SIMULATED prostate data: coordinate descent walks a staircase"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** Each coordinate update is an exact one-dimensional minimisation, so the path moves only
horizontally or vertically — a **staircase** toward the least-squares solution. With 8 standardised predictors, the
error relative to the closed form falls steadily with the number of sweeps: 0.38 after one sweep, 0.13 after five,
$8\times10^{-4}$ after 20, and machine precision ($10^{-15}$) after 100. Convergence is slower when predictors are
correlated (here `lcavol`, `lcp`, `gleason` and `pgg45`), because changing one weight changes the best value of the others — the staircase needs many small steps
along a tilted valley. That is the price; the reward is that every step is trivially cheap, needs no step size, and
adapts to non-smooth penalties like the lasso's $|w_j|$ where gradients do not exist.

✏️ **Your turn.** Replace `lcp` by `lweight` (correlation with `lcavol` about 0.28) in the two-predictor picture. Does
the staircase reach the solution in fewer steps? Explain using the shape of the contours.

## 📌 Key takeaways

- A **derivative** is a slope: the rate at which a score changes when a parameter is nudged. Five rules (power, exp, log,
  product, **chain**) give every derivative in 02-620; check them with **finite differences**. The logistic function
  satisfies $\sigma'(z) = \sigma(z)(1-\sigma(z))$.
- Optimum ⇒ derivative = 0: the **mean** minimises squared error (smooth, outlier-sensitive), the **median** minimises
  absolute error (robust, kinked).
- Maximise **log**-likelihoods: logs turn products into sums, keep the same maximiser, and avoid numerical underflow;
  use `logsumexp` for logs of sums.
- **Gradient descent/ascent** steps along $\mp\nabla f$; too large a step overshoots or diverges, too small a step crawls;
  stop when the improvement falls below $\varepsilon$. Centring/standardising features reshapes long thin valleys.
- **Convex** (bowl) or **concave** (dome) objectives have a single optimum — least squares, logistic regression (Mount
  Fuji). Non-convex ones (mixtures, k-means, neural networks) have local optima: use **random restarts**.
- With $\partial(\mathbf a^\top\mathbf w)/\partial\mathbf w = \mathbf a$ and $\partial(\mathbf w^\top A\mathbf w)/\partial\mathbf w = (A + A^\top)\mathbf w$, the gradient of $\|\mathbf y - X\mathbf w\|^2$ is $-2X^\top(\mathbf y - X\mathbf w)$; setting it
  to zero gives the normal equations.
- **Lagrange multipliers** handle constraints: maximising $\mathbf u^\top S\mathbf u$ with $\|\mathbf u\| = 1$ yields
  $S\mathbf u = \lambda\mathbf u$ (PCA); inequality constraints give a dual problem with multipliers $\alpha \ge 0$ that are
  non-zero only for tight constraints (SVM support vectors).
- **Coordinate descent** optimises one parameter at a time — the engine of the lasso and a cousin of EM and Gibbs sampling.

---
## ✅ Checkpoint

1. Differentiate $f(x) = 3x^4 - 2\ln x + e^{5x}$ by hand.
   <details><summary>Answer</summary>$f'(x) = 12x^3 - 2/x + 5e^{5x}$ (power rule, log rule, chain rule for $e^{5x}$).</details>
2. Show that $\frac{d}{dz}\ln\big(1 + e^{z}\big) = \sigma(z)$ and explain where this function appears in 02-620.
   <details><summary>Answer</summary>Chain rule: $\frac{1}{1+e^z}\cdot e^z = \frac{e^z}{1+e^z} = \frac{1}{1+e^{-z}} = \sigma(z)$. It is the second term of the logistic-regression log-likelihood $l(W) = \sum_l Y^l(w_0 + \sum_i w_iX^l_i) - \ln(1 + \exp(w_0 + \sum_i w_iX^l_i))$ (Section 5.2); differentiating it produces the $\hat P(Y^l = 1|X^l, W)$ term of the gradient.</details>
3. Five replicate measurements are 2, 3, 3, 4, 13. Which single number minimises the squared error, and which the absolute error? Which would you report?
   <details><summary>Answer</summary>Squared error → mean $= 25/5 = 5$; absolute error → median $= 3$. The 13 looks like a failed measurement; the median is robust to it, the mean is pulled up by it.</details>
4. Gradient ascent on $f(x) = -2(x - 1)^2$ uses $x_{t+1} = x_t + \lambda f'(x_t)$. For which step sizes does it converge, and for which does it oscillate?
   <details><summary>Answer</summary>$f'(x) = -4(x-1)$, so $x_{t+1} - 1 = (1 - 4\lambda)(x_t - 1)$. It converges iff $|1 - 4\lambda| < 1$, i.e. $0 < \lambda < 0.5$; it oscillates (overshoots) while converging for $0.25 < \lambda < 0.5$; $\lambda = 0.25$ jumps to the optimum in one step; $\lambda > 0.5$ diverges.</details>
5. Why does gradient ascent find the global maximum of the logistic-regression log-likelihood but not necessarily of a mixture-model log-likelihood?
   <details><summary>Answer</summary>The logistic log-likelihood is concave (Mount Fuji): any point with zero gradient is the global maximum. The mixture log-likelihood (a log of a sum) is not concave: it has several local maxima (e.g. label-swapped solutions), and gradient ascent stops at whichever is uphill from the start — hence random restarts.</details>
6. Using the vector rules, differentiate $E(\mathbf w) = (\mathbf y - X\mathbf w)^\top(\mathbf y - X\mathbf w) + \lambda\,\mathbf w^\top\mathbf w$ and solve $\nabla E = 0$.
   <details><summary>Answer</summary>$\nabla E = -2X^\top\mathbf y + 2X^\top X\mathbf w + 2\lambda\mathbf w = 0 \Rightarrow (X^\top X + \lambda I)\mathbf w = X^\top\mathbf y \Rightarrow \mathbf w = (X^\top X + \lambda I)^{-1}X^\top\mathbf y$ — ridge regression, the MAP estimate with a Gaussian prior (Section 6.4), invertible for $\lambda > 0$ (Module 03, Section 8).</details>
7. In the PCA Lagrangian $\mathbf u^\top S\mathbf u + \lambda(1 - \mathbf u^\top\mathbf u)$, why must the solution be the eigenvector with the *largest* eigenvalue rather than any eigenvector?
   <details><summary>Answer</summary>Every eigenvector satisfies the zero-gradient condition $S\mathbf u = \lambda\mathbf u$, but the objective at such a point is $\mathbf u^\top S\mathbf u = \lambda\mathbf u^\top\mathbf u = \lambda$. To *maximise* the variance we pick the largest $\lambda$ (the others are saddle points or the minimum).</details>
8. For the primal–dual toy of Section 7.2 with $b = 2$, give $x^\ast$, $\alpha^\ast$ and the common optimal value of primal and dual.
   <details><summary>Answer</summary>The constraint is active: $x^\ast = b = 2$, $\alpha^\ast = 2b = 4$, primal value $x^{\ast2} = 4$, dual value $-\alpha^{\ast2}/4 + \alpha^\ast b = -4 + 8 = 4$. Complementary slackness: $\alpha^\ast(x^\ast - b) = 4\cdot 0 = 0$.</details>

## 🧠 Questions & discussion

1. Squared error corresponds to Gaussian noise and absolute error to Laplace noise (Module 05). For RNA-seq counts, flow-cytometry intensities and qPCR Ct values, which error model seems most plausible, and what would go wrong with the other?
2. Gradient descent needed far fewer steps after centring the predictor. Why do neural-network practitioners standardise inputs, and what does this have to do with the eigenvalues of the Hessian?
3. Concavity of the logistic log-likelihood guarantees a unique maximum *if one exists*. What happens to the weights when ALL and AML patients are perfectly separated by one gene, and how does a Gaussian prior on the weights (MAP, an L2 penalty; Module 09) fix it?
4. Random restarts reduce the risk of reporting a local optimum but never eliminate it. How would you decide how many restarts are enough for EM on a single-cell data set with 20 clusters?
5. In the SVM, each training sample has its own multiplier $\alpha_i$, and most are zero at the optimum. What does that mean biologically if the samples are tumours and the features are genes? Which tumours 'define' the classifier?

## 🚀 Bonus — going further

- **Newton's method.** Replace the step size by the inverse curvature: $x_{t+1} = x_t - f'(x_t)/f''(x_t)$ (in many dimensions, $H^{-1}\nabla f$). Implement it for the logistic log-likelihood of Section 5.2 (Hessian $-\sum_l \sigma(1-\sigma)\,\mathbf x^l\mathbf x^{l\top}$) and count iterations versus gradient ascent. This is the IRLS algorithm statistics packages use for logistic regression.
- **Stochastic gradient descent.** Estimate the gradient from a random mini-batch of samples instead of all of them. Run it on the prostate regression with batch sizes 1, 8 and 67 and plot the noisy loss curves — the optimiser of Module 22.
- **Momentum.** Add a fraction of the previous step to the current one and rerun the raw-predictor case of 4.3. Does the zig-zag along the narrow valley disappear?
- **Duality for the soft-margin SVM.** Derive the dual of $\min\frac12 w^2 + C\sum_i\varepsilon_i$ for a 1-D toy with three points by hand (introduce multipliers for both $y_i(wx_i + b) \ge 1 - \varepsilon_i$ and $\varepsilon_i \ge 0$) and show where the bound $\alpha_i \le C$ comes from; check it with `svm_dual` of Section 7.3 — a warm-up for Module 11.
- **Further reading in this repository.** The [Fundamentals of learning and uncertainty](../../FundamentalsLearningUncertainty/) tutorial visualises likelihood and posterior landscapes, and the [Regularized linear regression](../../RegularizedLinearRegression/) tutorial implements coordinate descent with soft-thresholding for the lasso.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            04 · Calculus and optimization
Input:             toy functions; SIMULATED growth curve, wells, genotypes, mixture, tumour toy (SVM);
                   SIMULATED prostate table (lcavol → lpsa; 8 predictors);
                   SIMULATED leukaemia data (gene MYB for logistic regression; TCF3/CCND3 covariance)
Output:            derivatives (checked numerically), minimisers, gradient-descent paths, optima of constrained problems
Method:            finite differences; setting derivatives to zero; gradient descent/ascent with a stopping rule;
                   convexity checks (chords, Hessian eigenvalues); random restarts; Lagrangians; coordinate descent
Key equation:      ∇_w (y − Xw)ᵀ(y − Xw) = −2Xᵀ(y − Xw)   and   ∇_u [uᵀSu + λ(1 − uᵀu)] = 0 ⇒ Su = λu
Key parameters:    step size λ (η), tolerance ε, number of restarts, number of sweeps
What changed when I modified …  (e.g. step size 0.27 → 0.30; raw vs centred predictor; mixture weights 0.6/0.4 → 0.5/0.5)
One failure mode:  a step size above 2/(largest Hessian eigenvalue) diverges; non-convex objectives stop at local optima
Connection to the 02-620 topics (which idea of the lecture this illustrates):
                   logistic regression (gradient ascent, concavity); regression (normal equations, ridge);
                   SVM (Lagrangian duality, support vectors); PCA (constrained optimisation); lasso (coordinate descent)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the calculus and optimisation of its lectures on regression, logistic regression, support vector machines and PCA.
- Boyd S., Vandenberghe L. (2004) *Convex Optimization*. Cambridge University Press — ch. 3 (convex functions), ch. 5 (duality, Lagrangians, complementary slackness), §9.3 (gradient descent). Free PDF from the authors' web site.
- Petersen K. B., Pedersen M. S. (2012) *The Matrix Cookbook* (version of 15 November 2012), eqs. 69 and 81 — the two vector-calculus rules of Section 6.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — §7.3 (least squares derivation), §8.3 (logistic regression: gradient, Hessian, steepest descent, Newton), §12.2 (PCA).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §3.2, §4.4 (fitting logistic regression), §3.8.6 (pathwise coordinate optimisation).
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill — §4.4 (gradient descent and the delta rule).
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537 (the study the simulated leukaemia data imitate).
- Stamey T. A. *et al.* (1989) Prostate specific antigen in the diagnosis and treatment of adenocarcinoma of the prostate. II. Radical prostatectomy treated patients. *Journal of Urology* 141:1076–1083 (the study the simulated prostate table imitates).
- Friedman J., Hastie T., Höfling H., Tibshirani R. (2007) Pathwise coordinate optimization. *Annals of Applied Statistics* 1:302–332 (coordinate descent for the lasso; the `glmnet` package).
- Cortes C., Vapnik V. (1995) Support-vector networks. *Machine Learning* 20:273–297 (the SVM duality that Section 7 prepares).

## ➡️ Next

**[Module 05 · 🪙 Learning from data: likelihood, MLE and MAP](05_likelihood_mle_map.ipynb)** — you now know *how* to find
the top of a hill. Module 05 tells you *which hill to climb*: the likelihood of the data under a probability model, its
maximum (MLE) and the maximum of likelihood × prior (MAP) — the two-step recipe behind every model in 02-620.